# PRISM-3D Rebuild (Colab)
This notebook creates a clean `prism3d` package in Colab and validates the Colon-first data pipeline.

**If the runtime restarts, re-run the _Bootstrap package_ cell before importing `prism3d`.**


In [57]:
# --- Install dependencies (modern versions) ---
!pip -q install -U monai torchio nibabel einops scikit-image tqdm tensorboard matplotlib
# Optional robustness for NIfTI header inconsistencies:
!pip -q install -U SimpleITK
# Optional NSD implementation (we'll use later in evaluation):
!pip -q install -U surface-distance


In [58]:
# --- Imports + versions + GPU sanity ---
import os, sys, random
import numpy as np
import torch

import monai
import torchio as tio
import nibabel as nib
import einops
import skimage

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("monai:", monai.__version__)
print("torchio:", tio.__version__)
print("nibabel:", nib.__version__)
print("einops:", einops.__version__)
print("skimage:", skimage.__version__)

seed = 2026
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False


torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
monai: 1.5.2
torchio: 0.21.3
nibabel: 5.3.3
einops: 0.8.2
skimage: 0.26.0


## Bootstrap package (run this after every runtime restart)
Creates `/content/prism3d_project/prism3d/...` and makes it importable.

In [59]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/prism3d_project")  # change if you want Drive persistence
PKG_ROOT = PROJECT_ROOT / "prism3d"

for p in [PKG_ROOT, PKG_ROOT/"data", PKG_ROOT/"models", PKG_ROOT/"utils", PKG_ROOT/"engine", PKG_ROOT/"configs"]:
    p.mkdir(parents=True, exist_ok=True)

(PKG_ROOT/"__init__.py").write_text("", encoding="utf-8")
(PKG_ROOT/"data"/"__init__.py").write_text("", encoding="utf-8")
(PKG_ROOT/"models"/"__init__.py").write_text("", encoding="utf-8")

# sys.path must include the folder that CONTAINS the prism3d/ package
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import prism3d
print("Imported prism3d from:", prism3d.__file__)
print("sys.path[0]:", sys.path[0])


Imported prism3d from: /content/prism3d_project/prism3d/__init__.py
sys.path[0]: /content/prism3d_project


## Write data modules (`split.pkl` reader, transforms, dataset)
Uses **absolute imports** so it works reliably in notebooks.

In [60]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path("/content/prism3d_project")
PKG_ROOT = PROJECT_ROOT / "prism3d"

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip("\n"), encoding="utf-8")

# --- prism3d/data/splits.py ---
write_py(PKG_ROOT/"data"/"splits.py", '''
import os
import pickle
from typing import List, Tuple

def load_split_pkl(data_dir: str, split: str) -> List[Tuple[str, str]]:
    """Load PRISM-style split.pkl and return list of (image_abs_path, label_abs_path)."""
    pkl_path = os.path.join(data_dir, "split.pkl")
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"split.pkl not found at: {pkl_path}")

    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    # PRISM repo sometimes stores [dict] and uses the first element
    if isinstance(obj, list):
        if len(obj) == 0:
            raise ValueError("split.pkl contained an empty list.")
        obj = obj[0]

    if split not in obj:
        raise KeyError(f"Split '{split}' not found. Keys={list(obj.keys())}")

    d = obj[split]
    if not isinstance(d, dict):
        raise TypeError(f"Expected dict for split mapping, got {type(d)}")

    pairs = []
    for k in d.keys():
        img_rel, lbl_rel = d[k][0].strip("/"), d[k][1].strip("/")
        pairs.append((os.path.join(data_dir, img_rel), os.path.join(data_dir, lbl_rel)))
    return pairs
''')

# --- prism3d/data/transforms.py ---
write_py(PKG_ROOT/"data"/"transforms.py", '''
from dataclasses import dataclass
from typing import Tuple

from monai.transforms import (
    Compose,
    ScaleIntensityRanged,
    RandCropByPosNegLabeld,
    RandShiftIntensityd,
    NormalizeIntensityd,
    RandZoomd,
)

@dataclass(frozen=True)
class DatasetStats:
    intensity_range: Tuple[float, float]
    global_mean: float
    global_std: float
    target_label: int

# Matches constants from the PRISM repo dataloader
DATASET_STATS = {
    "colon":    DatasetStats(intensity_range=(-57, 175), global_mean=65.175035, global_std=32.651197, target_label=0),
    "pancreas": DatasetStats(intensity_range=(-39, 204), global_mean=68.45214,  global_std=63.422806, target_label=2),
    "lits":     DatasetStats(intensity_range=(-48, 163), global_mean=60.057533, global_std=40.198017, target_label=2),
    "kits":     DatasetStats(intensity_range=(-54, 247), global_mean=59.53867,  global_std=55.457336, target_label=2),
}

def build_monai_transforms(dataset: str, split: str):
    st = DATASET_STATS[dataset]
    if split == "train":
        return Compose([
            ScaleIntensityRanged(
                keys=["image"],
                a_min=st.intensity_range[0], a_max=st.intensity_range[1],
                b_min=st.intensity_range[0], b_max=st.intensity_range[1],
                clip=True,
            ),
            RandCropByPosNegLabeld(
                keys=["image", "label"],
                spatial_size=(128, 128, 128),
                label_key="label",
                pos=2,
                neg=0,
                num_samples=1,
            ),
            RandShiftIntensityd(keys=["image"], offsets=20, prob=0.5),
            NormalizeIntensityd(keys=["image"], subtrahend=st.global_mean, divisor=st.global_std),
            RandZoomd(
                keys=["image", "label"],
                prob=0.8,
                min_zoom=0.85,
                max_zoom=1.25,
                mode=["trilinear", "nearest"],
            ),
        ])
    else:
        return Compose([
            ScaleIntensityRanged(
                keys=["image"],
                a_min=st.intensity_range[0], a_max=st.intensity_range[1],
                b_min=st.intensity_range[0], b_max=st.intensity_range[1],
                clip=True,
            ),
            NormalizeIntensityd(keys=["image"], subtrahend=st.global_mean, divisor=st.global_std),
        ])
''')

# --- prism3d/data/dataset.py ---
write_py(PKG_ROOT/"data"/"dataset.py", '''
import os
import numpy as np
from torch.utils.data import Dataset
import torchio as tio

from prism3d.data.splits import load_split_pkl
from prism3d.data.transforms import DATASET_STATS, build_monai_transforms

try:
    import SimpleITK as sitk
    _HAS_SITK = True
except Exception:
    _HAS_SITK = False


class PrismCTDataset(Dataset):
    """Colon-first faithful dataset, matching PRISM repo behavior."""
    def __init__(
        self,
        dataset: str,
        data_dir: str,
        split: str,
        image_size: int = 128,
        torchio_transform: tio.Transform | None = None,
        threshold: int = 0,
    ):
        super().__init__()
        if dataset not in DATASET_STATS:
            raise KeyError(f"Unknown dataset '{dataset}'")
        if split not in ("train", "val", "test"):
            raise ValueError("split must be one of: train/val/test")

        self.dataset = dataset
        self.data_dir = data_dir
        self.split = split
        self.image_size = (image_size, image_size, image_size)
        self.threshold = threshold

        self.pairs = load_split_pkl(data_dir, split)
        self.stats = DATASET_STATS[dataset]
        self.monai_tf = build_monai_transforms(dataset, split)
        self.torchio_tf = torchio_transform
        self.crop_or_pad = tio.CropOrPad(mask_name="label", target_shape=self.image_size)

    def __len__(self):
        return len(self.pairs)

    def _load_subject(self, img_path: str, lbl_path: str) -> tio.Subject:
        if _HAS_SITK:
            sitk_image = sitk.ReadImage(img_path)
            sitk_label = sitk.ReadImage(lbl_path)

            if sitk_image.GetOrigin() != sitk_label.GetOrigin():
                sitk_image.SetOrigin(sitk_label.GetOrigin())
            if sitk_image.GetDirection() != sitk_label.GetDirection():
                sitk_image.SetDirection(sitk_label.GetDirection())
            if sitk_image.GetSpacing() != sitk_label.GetSpacing():
                sitk_label.SetSpacing(sitk_image.GetSpacing())

            return tio.Subject(
                image=tio.ScalarImage.from_sitk(sitk_image),
                label=tio.LabelMap.from_sitk(sitk_label),
            )

        return tio.Subject(
            image=tio.ScalarImage(img_path),
            label=tio.LabelMap(lbl_path),
        )

    def _binary_label(self, subject: tio.Subject) -> tio.Subject:
        lab = subject.label.data
        lab = (lab == self.stats.target_label).float()
        subject.label.data = lab
        return subject

    def __getitem__(self, idx: int):
        img_path, lbl_path = self.pairs[idx]
        if not os.path.exists(img_path):
            raise FileNotFoundError(img_path)
        if not os.path.exists(lbl_path):
            raise FileNotFoundError(lbl_path)

        subject = self._load_subject(img_path, lbl_path)

        if self.stats.target_label != 0:
            subject = self._binary_label(subject)

        if self.torchio_tf is not None:
            subject = self.torchio_tf(subject)

        if subject.label.data.sum().item() <= self.threshold and self.split == "train":
            new_idx = np.random.randint(0, len(self))
            return self.__getitem__(new_idx)

        if self.split == "train":
            out = self.monai_tf({"image": subject.image.data.clone(), "label": subject.label.data.clone()})
            if isinstance(out, list):
                out = out[0]
            return out["image"].float(), out["label"].float(), img_path

        subject = self.crop_or_pad(subject)
        out = self.monai_tf({"image": subject.image.data.clone()})
        return out["image"].float(), subject.label.data.clone().float(), img_path
''')

print("Wrote data modules into:", PKG_ROOT/"data")


Wrote data modules into: /content/prism3d_project/prism3d/data


## Create a tiny synthetic colon dataset (for validation only)

In [61]:
import pickle
import numpy as np
import nibabel as nib
from pathlib import Path

DATA_ROOT = Path("/content/prism_data/colon")
(DATA_ROOT/"imagesTr").mkdir(parents=True, exist_ok=True)
(DATA_ROOT/"labelsTr").mkdir(parents=True, exist_ok=True)

shape = (160, 160, 160)
img = (np.random.randn(*shape) * 20 + 60).astype(np.float32)

zz, yy, xx = np.ogrid[:shape[0], :shape[1], :shape[2]]
center = np.array(shape) // 2
radius = 25
lbl = (((zz-center[0])**2 + (yy-center[1])**2 + (xx-center[2])**2) <= radius**2).astype(np.uint8)

spacing = np.array([1.3, 1.1, 1.2], dtype=np.float32)
affine = np.eye(4, dtype=np.float32)
affine[0,0], affine[1,1], affine[2,2] = spacing

img_path = DATA_ROOT/"imagesTr"/"case_0000.nii.gz"
lbl_path = DATA_ROOT/"labelsTr"/"case_0000.nii.gz"
nib.save(nib.Nifti1Image(img, affine), str(img_path))
nib.save(nib.Nifti1Image(lbl, affine), str(lbl_path))

split_obj = [{
    "train": {0: ["/imagesTr/case_0000.nii.gz", "/labelsTr/case_0000.nii.gz"]},
    "val":   {0: ["/imagesTr/case_0000.nii.gz", "/labelsTr/case_0000.nii.gz"]},
    "test":  {0: ["/imagesTr/case_0000.nii.gz", "/labelsTr/case_0000.nii.gz"]},
}]
with open(DATA_ROOT/"split.pkl", "wb") as f:
    pickle.dump(split_obj, f)

print("Synthetic dataset ready:", DATA_ROOT)


Synthetic dataset ready: /content/prism_data/colon


## Validate dataset outputs (shapes must be 128³)

In [62]:
import torch
from torch.utils.data import DataLoader
import torchio as tio

from prism3d.data.dataset import PrismCTDataset

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

ds_train = PrismCTDataset(dataset="colon", data_dir=str(DATA_ROOT), split="train", torchio_transform=train_tio)
dl_train = DataLoader(ds_train, batch_size=2, shuffle=True, num_workers=0)

img, lab, path = next(iter(dl_train))
print("TRAIN image:", tuple(img.shape), img.dtype)
print("TRAIN label:", tuple(lab.shape), lab.dtype)
print("TRAIN image stats: min/max/mean =", float(img.min()), float(img.max()), float(img.mean()))
print("TRAIN label sum per item:", lab.view(lab.size(0), -1).sum(dim=1))

ds_val = PrismCTDataset(dataset="colon", data_dir=str(DATA_ROOT), split="val", torchio_transform=val_tio)
imgv, labv, _ = ds_val[0]
print("\nVAL image:", tuple(imgv.shape), imgv.dtype)
print("VAL label:", tuple(labv.shape), labv.dtype)
print("VAL label sum:", float(labv.sum()))


TRAIN image: (1, 1, 128, 128, 128) torch.float32
TRAIN label: (1, 1, 128, 128, 128) torch.float32
TRAIN image stats: min/max/mean = -1.8315421342849731 1.195886254310608 -0.15801118314266205
TRAIN label sum per item: metatensor([92091.])

VAL image: (1, 128, 128, 128) torch.float32
VAL label: (1, 128, 128, 128) torch.float32
VAL label sum: 111833.0


In [63]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'unet3d.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNormAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.norm = nn.InstanceNorm3d(out_ch, affine=True)
        self.act  = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.conv(x)))


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            ConvNormAct(in_ch, out_ch, 3, 1, 1),
            ConvNormAct(out_ch, out_ch, 3, 1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.down = ConvNormAct(in_ch, out_ch, k=3, s=2, p=1)
        self.conv = DoubleConv(out_ch, out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(self.down(x))


class Up(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False),
            nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.GELU(),
        )
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.pre(x)
        if x.shape[-3:] != skip.shape[-3:]:
            dz = skip.shape[-3] - x.shape[-3]
            dy = skip.shape[-2] - x.shape[-2]
            dx = skip.shape[-1] - x.shape[-1]
            x = F.pad(x, [dx//2, dx-dx//2, dy//2, dy-dy//2, dz//2, dz-dz//2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNetEncoder3D(nn.Module):
    # x0: (B, 32, 128,128,128)
    # x1: (B, 32, 64,64,64)
    # x2: (B, 64, 32,32,32)
    # x3: (B,128, 16,16,16)
    # x4: (B,384,  8, 8, 8)
    def __init__(self, in_ch: int = 1, base_ch: int = 32):
        super().__init__()
        self.stem = DoubleConv(in_ch, base_ch)
        self.d1   = Down(base_ch, base_ch)
        self.d2   = Down(base_ch, base_ch * 2)
        self.d3   = Down(base_ch * 2, base_ch * 4)
        self.d4   = Down(base_ch * 4, 384)

    def forward(self, x: torch.Tensor):
        x0 = self.stem(x)
        x1 = self.d1(x0)
        x2 = self.d2(x1)
        x3 = self.d3(x2)
        x4 = self.d4(x3)
        return x4, x3, x2, x1, x0


class UNetUpscaler3D(nn.Module):
    # output u0: (B, 32, 128,128,128)
    def __init__(self):
        super().__init__()
        self.u3 = Up(in_ch=384, skip_ch=128, out_ch=128)  # 8->16
        self.u2 = Up(in_ch=128, skip_ch=64,  out_ch=64)   # 16->32
        self.u1 = Up(in_ch=64,  skip_ch=32,  out_ch=32)   # 32->64
        self.u0 = Up(in_ch=32,  skip_ch=32,  out_ch=32)   # 64->128

    def forward(self, x4: torch.Tensor, skips):
        x0, x1, x2, x3 = skips
        y = self.u3(x4, x3)
        y = self.u2(y, x2)
        y = self.u1(y, x1)
        y = self.u0(y, x0)
        return y
''')

print('Wrote:', PKG_ROOT/'models'/'unet3d.py')

Wrote: /content/prism3d_project/prism3d/models/unet3d.py


In [64]:
import torch
from prism3d.models.unet3d import UNetEncoder3D, UNetUpscaler3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = UNetEncoder3D().to(device).eval()
up  = UNetUpscaler3D().to(device).eval()

x = torch.randn(1, 1, 128, 128, 128, device=device)
with torch.no_grad():
    x4, x3, x2, x1, x0 = enc(x)
    u0 = up(x4, [x0, x1, x2, x3])

print('x0:', tuple(x0.shape))
print('x1:', tuple(x1.shape))
print('x2:', tuple(x2.shape))
print('x3:', tuple(x3.shape))
print('x4:', tuple(x4.shape))
print('u0:', tuple(u0.shape))

x0: (1, 32, 128, 128, 128)
x1: (1, 32, 64, 64, 64)
x2: (1, 64, 32, 32, 32)
x3: (1, 128, 16, 16, 16)
x4: (1, 384, 8, 8, 8)
u0: (1, 32, 128, 128, 128)


In [65]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'vit3d.py', '''
import torch
import torch.nn as nn
from typing import List, Tuple

class PatchEmbed3D(nn.Module):
    def __init__(self, in_chans: int = 1, embed_dim: int = 768, patch_size: int = 16):
        super().__init__()
        self.proj = nn.Conv3d(
            in_chans, embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            padding=0,
            bias=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)  # (B,768,8,8,8) for 128^3


class MLP(nn.Module):
    def __init__(self, dim: int, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x


class SelfAttention(nn.Module):
    def __init__(self, dim: int = 768, num_heads: int = 12, qkv_bias: bool = True, drop: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, N, C)
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B,H,N,Hd)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B,H,N,N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v  # (B,H,N,Hd)
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.proj_drop(self.proj(out))
        return out


class TransformerBlock3D(nn.Module):
    def __init__(self, dim: int = 768, num_heads: int = 12, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim=dim, num_heads=num_heads, qkv_bias=True, drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim=dim, mlp_ratio=mlp_ratio, drop=drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViT3DEncoder(nn.Module):
    # patch embed -> (B,768,8,8,8) -> tokens (B,512,768)
    # 12 blocks (global attn)
    # neck -> (B,384,8,8,8)
    # returns out + fuse_feats (3 feature maps at indices 2,5,8)
    def __init__(
        self,
        in_chans: int = 1,
        embed_dim: int = 768,
        out_chans: int = 384,
        depth: int = 12,
        num_heads: int = 12,
        patch_size: int = 16,
        fuse_indices: Tuple[int, ...] = (2, 5, 8),
    ):
        super().__init__()
        self.patch_embed = PatchEmbed3D(in_chans=in_chans, embed_dim=embed_dim, patch_size=patch_size)

        self.pos_embed = nn.Parameter(torch.zeros(1, 512, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock3D(dim=embed_dim, num_heads=num_heads) for _ in range(depth)])
        self.fuse_indices = set(fuse_indices)

        self.neck = nn.Sequential(
            nn.Conv3d(embed_dim, out_chans, kernel_size=1, bias=False),
            nn.InstanceNorm3d(out_chans, affine=True),
            nn.GELU(),
            nn.Conv3d(out_chans, out_chans, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_chans, affine=True),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor):
        x = self.patch_embed(x)  # (B,768,8,8,8)
        B, C, D, H, W = x.shape
        assert (D, H, W) == (8, 8, 8), f'Expected 8x8x8 tokens, got {(D,H,W)}'

        tokens = x.flatten(2).transpose(1, 2)  # (B,512,768)
        tokens = tokens + self.pos_embed

        fuse_feats: List[torch.Tensor] = []
        for i, blk in enumerate(self.blocks):
            tokens = blk(tokens)
            if i in self.fuse_indices:
                fm = tokens.transpose(1, 2).reshape(B, C, D, H, W).contiguous()
                fuse_feats.append(fm)

        fm_final = tokens.transpose(1, 2).reshape(B, C, D, H, W).contiguous()
        out = self.neck(fm_final)  # (B,384,8,8,8)
        return out, fuse_feats
''')

print('Wrote:', PKG_ROOT/'models'/'vit3d.py')

Wrote: /content/prism3d_project/prism3d/models/vit3d.py


In [66]:
import torch
from prism3d.models.vit3d import ViT3DEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
vit = ViT3DEncoder().to(device).eval()

x = torch.randn(1, 1, 128, 128, 128, device=device)
with torch.no_grad():
    out, fuse_feats = vit(x)

print('ViT out:', tuple(out.shape))
for i, f in enumerate(fuse_feats):
    print(f'fuse_feats[{i}]:', tuple(f.shape))

ViT out: (1, 384, 8, 8, 8)
fuse_feats[0]: (1, 768, 8, 8, 8)
fuse_feats[1]: (1, 768, 8, 8, 8)
fuse_feats[2]: (1, 768, 8, 8, 8)


In [67]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared caches.')

Cleared caches.


In [68]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'hybrid_encoder3d.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F

from prism3d.models.unet3d import UNetEncoder3D, UNetUpscaler3D
from prism3d.models.vit3d import ViT3DEncoder


class ConvNormAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 1, s: int = 1, p: int = 0):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.norm = nn.InstanceNorm3d(out_ch, affine=True)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.conv(x)))


class HybridImageEncoder3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = UNetEncoder3D(in_ch=1, base_ch=32)
        self.vit = ViT3DEncoder(in_chans=1)

        # project ViT fuse features (768@8^3) to match skip channels
        self.proj_x3 = ConvNormAct(768, 128, k=1, s=1, p=0)  # -> 128 then upsample to 16^3
        self.proj_x2 = ConvNormAct(768, 64,  k=1, s=1, p=0)  # -> 64 then upsample to 32^3
        self.proj_x1 = ConvNormAct(768, 32,  k=1, s=1, p=0)  # -> 32 then upsample to 64^3

        # bottleneck fusion (x4 + vit_out, both 384@8^3)
        self.cnn_bottleneck = ConvNormAct(384, 384, k=1, s=1, p=0)
        self.fuse_out = ConvNormAct(384, 384, k=3, s=1, p=1)

        # upscaler to 128^3 embedding for mask decoding later
        self.upscaler = UNetUpscaler3D()

    def forward(self, x: torch.Tensor):
        # CNN
        x4, x3, x2, x1, x0 = self.cnn(x)

        # ViT
        vit_out, fuse_feats = self.vit(x)
        if len(fuse_feats) != 3:
            raise RuntimeError(f'Expected 3 fuse feats, got {len(fuse_feats)}')

        # Fuse skips (add after projecting + upsampling)
        f3 = self.proj_x3(fuse_feats[0])
        f2 = self.proj_x2(fuse_feats[1])
        f1 = self.proj_x1(fuse_feats[2])

        f3 = F.interpolate(f3, scale_factor=2, mode='trilinear', align_corners=False)  # 8->16
        f2 = F.interpolate(f2, scale_factor=4, mode='trilinear', align_corners=False)  # 8->32
        f1 = F.interpolate(f1, scale_factor=8, mode='trilinear', align_corners=False)  # 8->64

        x3_fused = x3 + f3
        x2_fused = x2 + f2
        x1_fused = x1 + f1
        x0_fused = x0

        # Fuse bottleneck
        x4p = self.cnn_bottleneck(x4)
        image_embedding = self.fuse_out(vit_out + x4p)  # (B,384,8,8,8)

        # Upscaled embedding (used later by mask decoder)
        upscaled_embedding = self.upscaler(image_embedding, [x0_fused, x1_fused, x2_fused, x3_fused])  # (B,32,128^3)

        feature_list = [x0_fused, x1_fused, x2_fused, x3_fused]
        return image_embedding, upscaled_embedding, feature_list
''')

print('Wrote:', PKG_ROOT/'models'/'hybrid_encoder3d.py')

Wrote: /content/prism3d_project/prism3d/models/hybrid_encoder3d.py


In [69]:
import torch
from prism3d.models.hybrid_encoder3d import HybridImageEncoder3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
hyb = HybridImageEncoder3D().to(device).eval()

x = torch.randn(1, 1, 128, 128, 128, device=device)
with torch.no_grad():
    image_emb, up_emb, feats = hyb(x)

print('image_embedding:', tuple(image_emb.shape))
print('upscaled_embedding:', tuple(up_emb.shape))
print('feature_list lens:', len(feats))
for i, f in enumerate(feats):
    print(f'feat[{i}]:', tuple(f.shape))

image_embedding: (1, 384, 8, 8, 8)
upscaled_embedding: (1, 32, 128, 128, 128)
feature_list lens: 4
feat[0]: (1, 32, 128, 128, 128)
feat[1]: (1, 32, 64, 64, 64)
feat[2]: (1, 64, 32, 32, 32)
feat[3]: (1, 128, 16, 16, 16)


In [70]:
import torch
from torch.utils.data import DataLoader
import torchio as tio

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.hybrid_encoder3d import HybridImageEncoder3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

DATA_ROOT = '/content/prism_data/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
dl = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))
img = img.to(device)

hyb = HybridImageEncoder3D().to(device).eval()
with torch.no_grad():
    image_emb, up_emb, feats = hyb(img)

print('input:', tuple(img.shape))
print('image_embedding:', tuple(image_emb.shape))
print('upscaled_embedding:', tuple(up_emb.shape))

input: (1, 1, 128, 128, 128)
image_embedding: (1, 384, 8, 8, 8)
upscaled_embedding: (1, 32, 128, 128, 128)


In [71]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared caches.')

Cleared caches.


In [72]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'twoway_transformer.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class MLPBlock(nn.Module):
    def __init__(self, dim: int, mlp_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(dim, mlp_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(mlp_dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))


class Attention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError('dim must be divisible by num_heads')
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        B, Lq, C = q.shape
        _, Lk, _ = k.shape

        q = self.q_proj(q)
        k = self.k_proj(k)
        v = self.v_proj(v)

        q = q.view(B, Lq, self.num_heads, self.head_dim).transpose(1, 2)  # (B,H,Lq,Hd)
        k = k.view(B, Lk, self.num_heads, self.head_dim).transpose(1, 2)  # (B,H,Lk,Hd)
        v = v.view(B, Lk, self.num_heads, self.head_dim).transpose(1, 2)  # (B,H,Lk,Hd)

        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)       # (B,H,Lq,Hd)
        out = out.transpose(1, 2).contiguous().view(B, Lq, C)              # (B,Lq,C)
        out = self.out_proj(out)
        return out


class TwoWayAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_dim: int):
        super().__init__()
        self.self_attn = Attention(dim, num_heads)
        self.cross_attn_token_to_image = Attention(dim, num_heads)
        self.cross_attn_image_to_token = Attention(dim, num_heads)

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)
        self.norm4 = nn.LayerNorm(dim)

        self.mlp = MLPBlock(dim, mlp_dim)

    def forward(
        self,
        queries: torch.Tensor,
        keys: torch.Tensor,
        query_pe: torch.Tensor,
        key_pe: torch.Tensor,
    ):
        # 1) self-attn on queries
        q = self.norm1(queries)
        queries = queries + self.self_attn(q + query_pe, q + query_pe, q)

        # 2) cross-attn: queries attend to image
        q = self.norm2(queries)
        k = self.norm2(keys)
        queries = queries + self.cross_attn_token_to_image(q + query_pe, k + key_pe, k)

        # 3) mlp on queries
        q = self.norm3(queries)
        queries = queries + self.mlp(q)

        # 4) cross-attn: image attends to queries (updates keys)
        k = self.norm4(keys)
        qn = self.norm4(queries)
        keys = keys + self.cross_attn_image_to_token(k + key_pe, qn + query_pe, qn)

        return queries, keys


class TwoWayTransformer(nn.Module):
    def __init__(self, depth: int = 2, dim: int = 384, num_heads: int = 8, mlp_dim: int = 2048):
        super().__init__()
        self.layers = nn.ModuleList([TwoWayAttentionBlock(dim, num_heads, mlp_dim) for _ in range(depth)])
        self.final_attn_token_to_image = Attention(dim, num_heads)
        self.norm_final = nn.LayerNorm(dim)

    def forward(
        self,
        image_tokens: torch.Tensor,
        image_pe: torch.Tensor,
        point_tokens: torch.Tensor,
        point_pe: torch.Tensor,
    ):
        queries = point_tokens
        keys = image_tokens

        for layer in self.layers:
            queries, keys = layer(queries, keys, point_pe, image_pe)

        # final token->image attention
        q = self.norm_final(queries)
        k = self.norm_final(keys)
        queries = queries + self.final_attn_token_to_image(q + point_pe, k + image_pe, k)

        return queries, keys
''')

print('Wrote:', PKG_ROOT/'models'/'twoway_transformer.py')

Wrote: /content/prism3d_project/prism3d/models/twoway_transformer.py


In [73]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'prompt_encoder3d.py', '''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple


class PositionEmbeddingRandom3D(nn.Module):
    def __init__(self, num_pos_feats: int = 192, scale: float = 1.0):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.scale = scale
        self.register_buffer('gaussian_matrix', torch.randn(3, num_pos_feats) * scale)

    def _pe(self, coords: torch.Tensor) -> torch.Tensor:
        # (.., 3) @ (3, F) -> (.., F)
        proj = 2 * math.pi * (coords @ self.gaussian_matrix)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

    def forward_with_coords(self, coords_zyx: torch.Tensor, image_size: int) -> torch.Tensor:
        coords = coords_zyx / float(image_size - 1)
        coords = coords.clamp(0.0, 1.0)
        return self._pe(coords)

    def forward_grid(self, grid_size: Tuple[int, int, int], device: torch.device) -> torch.Tensor:
        D, H, W = grid_size
        z = torch.linspace(0.0, 1.0, steps=D, device=device)
        y = torch.linspace(0.0, 1.0, steps=H, device=device)
        x = torch.linspace(0.0, 1.0, steps=W, device=device)
        zz, yy, xx = torch.meshgrid(z, y, x, indexing='ij')
        coords = torch.stack([zz, yy, xx], dim=-1).view(-1, 3)  # (DHW,3)
        pe = self._pe(coords).unsqueeze(0)                      # (1,DHW,384)
        return pe


class MaskDownscaler3D(nn.Module):
    def __init__(self, embed_dim: int = 384):
        super().__init__()
        ch = 32
        self.net = nn.Sequential(
            nn.Conv3d(1, ch, kernel_size=2, stride=2),          # 128->64
            nn.InstanceNorm3d(ch, affine=True),
            nn.GELU(),

            nn.Conv3d(ch, ch, kernel_size=2, stride=2),         # 64->32
            nn.InstanceNorm3d(ch, affine=True),
            nn.GELU(),

            nn.Conv3d(ch, ch * 2, kernel_size=2, stride=2),     # 32->16
            nn.InstanceNorm3d(ch * 2, affine=True),
            nn.GELU(),

            nn.Conv3d(ch * 2, ch * 4, kernel_size=2, stride=2), # 16->8
            nn.InstanceNorm3d(ch * 4, affine=True),
            nn.GELU(),

            nn.Conv3d(ch * 4, embed_dim, kernel_size=1),
        )

    def forward(self, mask: torch.Tensor) -> torch.Tensor:
        return self.net(mask)


class PromptEncoder3D(nn.Module):
    def __init__(self, embed_dim: int = 384, image_size: int = 128):
        super().__init__()
        self.embed_dim = embed_dim
        self.image_size = image_size

        self.pe_layer = PositionEmbeddingRandom3D(num_pos_feats=embed_dim // 2)

        # point type embeddings
        self.pos_point_embed = nn.Embedding(1, embed_dim)
        self.neg_point_embed = nn.Embedding(1, embed_dim)
        self.not_a_point_embed = nn.Embedding(1, embed_dim)

        # box corner embeddings
        self.corner0_embed = nn.Embedding(1, embed_dim)
        self.corner1_embed = nn.Embedding(1, embed_dim)

        # mask embeddings
        self.mask_downscaler = MaskDownscaler3D(embed_dim=embed_dim)
        self.no_mask_embed = nn.Embedding(1, embed_dim)

    def get_dense_pe(self, device: torch.device) -> torch.Tensor:
        return self.pe_layer.forward_grid((8, 8, 8), device=device)  # (1,512,384)

    def forward(
        self,
        points: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        boxes: Optional[torch.Tensor] = None,
        masks: Optional[torch.Tensor] = None,
    ):
        device = None
        B = 1

        sparse_embeddings = []

        # Points
        if points is not None:
            coords, labels = points
            device = coords.device
            B = coords.shape[0]

            pe = self.pe_layer.forward_with_coords(coords, self.image_size)  # (B,N,384)
            # add label embeddings
            pos = self.pos_point_embed.weight[0]
            neg = self.neg_point_embed.weight[0]
            pad = self.not_a_point_embed.weight[0]

            out = pe.clone()
            out[labels == 1] = out[labels == 1] + pos
            out[labels == 0] = out[labels == 0] + neg
            out[labels == -1] = out[labels == -1] + pad

            sparse_embeddings.append(out)

        # Boxes
        if boxes is not None:
            if boxes.dim() == 2 and boxes.shape[1] == 6:
                corner0 = boxes[:, 0:3]
                corner1 = boxes[:, 3:6]
            elif boxes.dim() == 3 and boxes.shape[1] == 2 and boxes.shape[2] == 3:
                corner0 = boxes[:, 0, :]
                corner1 = boxes[:, 1, :]
            else:
                raise ValueError('boxes must be (B,6) or (B,2,3)')

            device = boxes.device if device is None else device
            B = boxes.shape[0]

            c0_pe = self.pe_layer.forward_with_coords(corner0.unsqueeze(1), self.image_size)[:, 0, :]  # (B,384)
            c1_pe = self.pe_layer.forward_with_coords(corner1.unsqueeze(1), self.image_size)[:, 0, :]  # (B,384)

            c0 = c0_pe + self.corner0_embed.weight[0]
            c1 = c1_pe + self.corner1_embed.weight[0]
            sparse_embeddings.append(torch.stack([c0, c1], dim=1))  # (B,2,384)

        if len(sparse_embeddings) == 0:
            # no sparse prompts => return empty tokens
            if device is None:
                device = torch.device('cpu')
            sparse = torch.zeros((B, 0, self.embed_dim), device=device)
        else:
            sparse = torch.cat(sparse_embeddings, dim=1)  # (B,Ns,384)

        # Dense mask prompt
        if masks is not None:
            dense = self.mask_downscaler(masks)  # (B,384,8,8,8)
        else:
            if device is None:
                device = torch.device('cpu')
            dense = self.no_mask_embed.weight[0].view(1, self.embed_dim, 1, 1, 1).repeat(B, 1, 8, 8, 8)

        return sparse, dense
''')

print('Wrote:', PKG_ROOT/'models'/'prompt_encoder3d.py')

Wrote: /content/prism3d_project/prism3d/models/prompt_encoder3d.py


In [74]:
import torch
from prism3d.models.prompt_encoder3d import PromptEncoder3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pe = PromptEncoder3D(embed_dim=384, image_size=128).to(device).eval()

# 3 points: pos, neg, pad
coords = torch.tensor([[[64, 64, 64],
                        [80, 80, 80],
                        [0, 0, 0]]], device=device).float()
labels = torch.tensor([[1, 0, -1]], device=device).long()

# one box: z0,y0,x0,z1,y1,x1
boxes = torch.tensor([[40, 40, 40, 90, 90, 90]], device=device).float()

# dummy mask prompt
masks = torch.zeros((1, 1, 128, 128, 128), device=device)
masks[:, :, 50:78, 50:78, 50:78] = 1.0

with torch.no_grad():
    sparse, dense = pe(points=(coords, labels), boxes=boxes, masks=masks)

print('sparse:', tuple(sparse.shape))  # expected (1, 3 + 2, 384) = (1,5,384)
print('dense:', tuple(dense.shape))    # expected (1,384,8,8,8)

sparse: (1, 5, 384)
dense: (1, 384, 8, 8, 8)


In [75]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'mask_decoder3d.py', '''
import torch
import torch.nn as nn
from typing import Tuple

from prism3d.models.twoway_transformer import TwoWayTransformer


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, num_layers: int):
        super().__init__()
        layers = []
        dims = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.GELU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MaskDecoder3D(nn.Module):
    def __init__(self, embed_dim: int = 384, num_multimask_outputs: int = 3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_multimask_outputs = num_multimask_outputs
        self.num_mask_tokens = num_multimask_outputs + 1

        self.transformer = TwoWayTransformer(depth=2, dim=embed_dim, num_heads=8, mlp_dim=2048)

        self.iou_token = nn.Embedding(1, embed_dim)
        self.mask_tokens = nn.Embedding(self.num_mask_tokens, embed_dim)

        self.iou_prediction_head = MLP(embed_dim, 256, self.num_mask_tokens, num_layers=3)

        # hypernetworks map (384) -> (32) to match upscaled_embedding channels
        self.hyper_mlps = nn.ModuleList([MLP(embed_dim, 256, 32, num_layers=3) for _ in range(self.num_mask_tokens)])

    def forward(
        self,
        image_embedding: torch.Tensor,          # (B,384,8,8,8)
        image_pe: torch.Tensor,                # (B,512,384)
        sparse_prompt_embeddings: torch.Tensor,# (B,Ns,384)
        dense_prompt_embeddings: torch.Tensor, # (B,384,8,8,8)
        upscaled_embedding: torch.Tensor,      # (B,32,128,128,128)
        multimask_output: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B = image_embedding.shape[0]

        # Add dense prompt to image embedding then flatten to tokens
        src = image_embedding + dense_prompt_embeddings
        src_tokens = src.flatten(2).transpose(1, 2)  # (B,512,384)

        # output tokens (iou + mask tokens)
        out_tokens = torch.cat([self.iou_token.weight, self.mask_tokens.weight], dim=0)  # (1+T,384)
        out_tokens = out_tokens.unsqueeze(0).repeat(B, 1, 1)                            # (B,1+T,384)

        # concat with sparse prompts
        tokens = torch.cat([out_tokens, sparse_prompt_embeddings], dim=1)               # (B,1+T+Ns,384)

        # positional encoding for tokens: none (prompts already contain PE), so zeros
        token_pe = torch.zeros_like(tokens)

        hs, _ = self.transformer(
            image_tokens=src_tokens,
            image_pe=image_pe,
            point_tokens=tokens,
            point_pe=token_pe,
        )

        iou_out = hs[:, 0, :]                                  # (B,384)
        mask_out = hs[:, 1:1 + self.num_mask_tokens, :]        # (B,T,384)

        iou_pred = self.iou_prediction_head(iou_out)           # (B,T)

        # hypernetworks -> masks
        masks = []
        for i in range(self.num_mask_tokens):
            hyper = self.hyper_mlps[i](mask_out[:, i, :])      # (B,32)
            m = (upscaled_embedding * hyper[:, :, None, None, None]).sum(dim=1)  # (B,128,128,128)
            masks.append(m)

        masks = torch.stack(masks, dim=1)                      # (B,T,128,128,128)

        if multimask_output:
            # return last 3 (like SAM-style multimask)
            return masks[:, 1:, ...], iou_pred[:, 1:]
        return masks[:, :1, ...], iou_pred[:, :1]
''')

print('Wrote:', PKG_ROOT/'models'/'mask_decoder3d.py')

Wrote: /content/prism3d_project/prism3d/models/mask_decoder3d.py


In [76]:
import torch
from prism3d.models.hybrid_encoder3d import HybridImageEncoder3D
from prism3d.models.prompt_encoder3d import PromptEncoder3D
from prism3d.models.mask_decoder3d import MaskDecoder3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

hyb = HybridImageEncoder3D().to(device).eval()
pe  = PromptEncoder3D(embed_dim=384, image_size=128).to(device).eval()
md  = MaskDecoder3D(embed_dim=384, num_multimask_outputs=3).to(device).eval()

# dummy input volume
x = torch.randn(1, 1, 128, 128, 128, device=device)

# simple prompts
coords = torch.tensor([[[64, 64, 64],
                        [80, 80, 80]]], device=device).float()
labels = torch.tensor([[1, 0]], device=device).long()
boxes  = torch.tensor([[40, 40, 40, 90, 90, 90]], device=device).float()

with torch.no_grad():
    image_emb, up_emb, _ = hyb(x)
    sparse, dense = pe(points=(coords, labels), boxes=boxes, masks=None)
    image_pe = pe.get_dense_pe(device=device).repeat(1, 1, 1)  # (1,512,384)
    masks, scores = md(
        image_embedding=image_emb,
        image_pe=image_pe,
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        upscaled_embedding=up_emb,
        multimask_output=True,
    )

print('image_emb:', tuple(image_emb.shape))
print('up_emb:', tuple(up_emb.shape))
print('sparse:', tuple(sparse.shape))
print('dense:', tuple(dense.shape))
print('masks:', tuple(masks.shape))    # expected (1,3,128,128,128)
print('scores:', tuple(scores.shape))  # expected (1,3)

image_emb: (1, 384, 8, 8, 8)
up_emb: (1, 32, 128, 128, 128)
sparse: (1, 4, 384)
dense: (1, 384, 8, 8, 8)
masks: (1, 3, 128, 128, 128)
scores: (1, 3)


In [77]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared caches.')

Cleared caches.


In [78]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_utils.py', '''
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.losses import DiceCELoss, DiceLoss


def bbox_from_mask_zyx(mask_volume: torch.Tensor, mode: str = 'train', dynamic: bool = False, max_diff: int = 10) -> torch.Tensor:
    bboxes = []
    for vol in mask_volume:
        i_any = vol.any(dim=2).any(dim=1)
        j_any = vol.any(dim=2).any(dim=0)
        k_any = vol.any(dim=1).any(dim=0)

        # assume at least one voxel is foreground
        i_min, i_max = torch.where(i_any)[0][[0, -1]]
        j_min, j_max = torch.where(j_any)[0][[0, -1]]
        k_min, k_max = torch.where(k_any)[0][[0, -1]]

        if dynamic and mode == 'train':
            diff_ = torch.randint(low=0, high=max_diff, size=(6,), device=vol.device)
            # clamp like repo (assumes 128 cube => max index 126 in their code)
            if max(0, int(i_min) - int(diff_[0])) < min(int(i_max) + int(diff_[1]), 126):
                i_min = max(0, int(i_min) - int(diff_[0]))
                i_max = min(int(i_max) + int(diff_[1]), 126)
            if max(0, int(j_min) - int(diff_[2])) < min(int(j_max) + int(diff_[3]), 126):
                j_min = max(0, int(j_min) - int(diff_[2]))
                j_max = min(int(j_max) + int(diff_[3]), 126)
            if max(0, int(k_min) - int(diff_[4])) < min(int(k_max) + int(diff_[5]), 126):
                k_min = max(0, int(k_min) - int(diff_[4]))
                k_max = min(int(k_max) + int(diff_[5]), 126)

        # +1 exclusive end
        bb = torch.tensor([int(i_min), int(j_min), int(k_min), int(i_max) + 1, int(j_max) + 1, int(k_max) + 1], device=vol.device)
        bboxes.append(bb)

    return torch.stack(bboxes, dim=0)  # (B,6)


def sample_click_points(prev_prob: torch.Tensor, label: torch.Tensor, num_clicks: int, dynamic: bool, mode: str):
    B = label.shape[0]
    pred_masks = (prev_prob > 0.5)
    true_masks = (label > 0)

    fn_masks = torch.logical_and(true_masks, torch.logical_not(pred_masks))
    fp_masks = torch.logical_and(torch.logical_not(true_masks), pred_masks)
    to_point_mask = torch.logical_or(fn_masks, fp_masks)  # (B,1,D,H,W)

    # choose dynamic_size based on smallest number of candidates across batch (repo behavior)
    points_list = [int(torch.argwhere(to_point_mask[i]).shape[0]) for i in range(B)]
    points_min = min(points_list) if len(points_list) > 0 else 0

    if points_min == 0:
        # no errors -> return empty
        coords = torch.zeros((B, 0, 3), device=label.device).float()
        labs = torch.zeros((B, 0), device=label.device).long()
        return coords, labs

    click_size = points_min if num_clicks > points_min else num_clicks
    if dynamic and mode == 'train':
        dynamic_size = random.randint(1, max(1, int(click_size)))
    else:
        dynamic_size = int(click_size)

    batch_coords = []
    batch_labs = []
    for i in range(B):
        pts = torch.argwhere(to_point_mask[i])  # rows: (idx0, z, y, x)
        # sample without replacement
        choice = torch.randperm(pts.shape[0], device=pts.device)[:dynamic_size]
        pts_sel = pts[choice]

        coords_i = []
        labs_i = []
        for p in pts_sel:
            zyx = p[1:].reshape(1, 3)  # (1,3)
            is_pos = bool(fn_masks[i, 0, int(p[1]), int(p[2]), int(p[3])].item())
            coords_i.append(zyx)
            labs_i.append(torch.tensor([1 if is_pos else 0], device=label.device).long())

        coords_i = torch.cat(coords_i, dim=0).unsqueeze(0)  # (1,N,3)
        labs_i = torch.cat(labs_i, dim=0).unsqueeze(0)      # (1,N)
        batch_coords.append(coords_i)
        batch_labs.append(labs_i)

    coords = torch.cat(batch_coords, dim=0).float()  # (B,N,3)
    labs = torch.cat(batch_labs, dim=0).long()       # (B,N)
    return coords, labs


class PrismLosses3D(nn.Module):
    def __init__(self, boundary_kernel_size: int = 5, device: str = 'cuda'):
        super().__init__()
        pad = int((boundary_kernel_size - 1) / 2)
        self.pool = nn.AvgPool3d((boundary_kernel_size, boundary_kernel_size, 1), stride=1, padding=(pad, pad, 0)).to(device)

        self.loss_boundary = nn.MSELoss()
        self.loss_seg = DiceCELoss(sigmoid=True, squared_pred=True, reduction='mean')
        self.loss_val_dice = DiceLoss(sigmoid=True, reduction='none')

    def forward(self, mask_logits: torch.Tensor, label: torch.Tensor, pred_score: torch.Tensor) -> torch.Tensor:
        mask_prob = torch.sigmoid(mask_logits)

        seg_edge = (label - self.pool(label)).abs()
        mask_edge = (mask_prob - self.pool(mask_prob)).abs()

        # target_dice per sample, robust to monai output shape
        pred_dice_score_loss = 0.0
        for b in range(mask_logits.shape[0]):
            dl = self.loss_val_dice(mask_logits[b:b+1], label[b:b+1])
            target_dice = 1.0 - dl.mean()
            pred_dice_score_loss = pred_dice_score_loss + self.loss_boundary(pred_score[b], target_dice)

        loss = self.loss_seg(mask_logits, label) + 10.0 * self.loss_boundary(mask_edge, seg_edge)
        loss = loss + pred_dice_score_loss
        return loss
''')

print('Wrote:', PKG_ROOT/'engine'/'interactive_utils.py')

Wrote: /content/prism3d_project/prism3d/engine/interactive_utils.py


In [79]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_runner.py', '''
import torch
from prism3d.engine.interactive_utils import sample_click_points, bbox_from_mask_zyx, PrismLosses3D


class InteractiveRunner3D:
    def __init__(
        self,
        model,
        device: str = 'cuda',
        iter_nums: int = 11,
        num_clicks_train: int = 50,
        num_clicks_val: int = 10,
        dynamic_clicks: bool = True,
        use_box: bool = True,
        dynamic_box: bool = False,
        boundary_kernel_size: int = 5,
        multiple_outputs: bool = True,
    ):
        self.model = model
        self.device = device

        self.iter_nums = iter_nums
        self.num_clicks_train = num_clicks_train
        self.num_clicks_val = num_clicks_val
        self.dynamic_clicks = dynamic_clicks

        self.use_box = use_box
        self.dynamic_box = dynamic_box
        self.multiple_outputs = multiple_outputs

        self.losses = PrismLosses3D(boundary_kernel_size=boundary_kernel_size, device=device)

        # history like repo (useful later for refinement module)
        self.click_points_hist = []
        self.click_labels_hist = []

    def run(self, image: torch.Tensor, label: torch.Tensor, train: bool = True):
        image = image.to(self.device)
        label = label.to(self.device)

        self.click_points_hist = []
        self.click_labels_hist = []

        # compute image features once
        image_emb, up_emb, _ = self.model.image_encoder(image)

        prev_logits = torch.zeros_like(label, dtype=torch.float32, device=self.device)

        total = 0.0
        for it in range(self.iter_nums):
            prev_prob = torch.sigmoid(prev_logits) if it > 0 else prev_logits

            mode = 'train' if train else 'validation'
            num_clicks = self.num_clicks_train if train else self.num_clicks_val

            coords, labs = sample_click_points(prev_prob, label, num_clicks=num_clicks, dynamic=self.dynamic_clicks, mode=mode)

            self.click_points_hist.append(coords)
            self.click_labels_hist.append(labs)

            # bbox from GT label like repo (optional)
            box = None
            if self.use_box:
                gt = (label[:, 0] > 0)
                box = bbox_from_mask_zyx(gt, mode=mode, dynamic=self.dynamic_box).float()  # (B,6)

            # model step
            masks, scores = self.model.step(
                image_embedding=image_emb,
                upscaled_embedding=up_emb,
                prev_mask_prob=prev_prob,
                points=(coords, labs),
                boxes=box,
                multimask_output=True,
            )  # masks: (B,3,128,128,128), scores: (B,3)

            if self.multiple_outputs:
                best_idx = torch.argmax(scores, dim=1)  # (B,)
                best = torch.stack([masks[b, best_idx[b]] for b in range(masks.shape[0])], dim=0).unsqueeze(1)  # (B,1,128,128,128)
            else:
                best = masks[:, 0:1]

            if train:
                loss = 0.0
                # sum losses for each output like repo
                for k in range(masks.shape[1]):
                    mk = masks[:, k:k+1]
                    sk = scores[:, k]
                    loss = loss + self.losses(mk, label, sk)
                total = total + loss
            else:
                # val-style: return dice of best
                prob = torch.sigmoid(best)
                pred = (prob > 0.5).float()
                gt = (label > 0).float()
                inter = (pred * gt).sum(dim=(1,2,3,4))
                denom = pred.sum(dim=(1,2,3,4)) + gt.sum(dim=(1,2,3,4)) + 1e-8
                dice = (2.0 * inter / denom).mean()
                total = total + dice

            prev_logits = best

        mean = total / float(self.iter_nums)
        return mean, prev_logits
''')

print('Wrote:', PKG_ROOT/'engine'/'interactive_runner.py')

Wrote: /content/prism3d_project/prism3d/engine/interactive_runner.py


In [80]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'prism_sam3d.py', '''
import torch
import torch.nn as nn

from prism3d.models.hybrid_encoder3d import HybridImageEncoder3D
from prism3d.models.prompt_encoder3d import PromptEncoder3D
from prism3d.models.mask_decoder3d import MaskDecoder3D


class PrismSAM3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_encoder = HybridImageEncoder3D()
        self.prompt_encoder = PromptEncoder3D(embed_dim=384, image_size=128)
        self.mask_decoder = MaskDecoder3D(embed_dim=384, num_multimask_outputs=3)

    def step(
        self,
        image_embedding: torch.Tensor,     # (B,384,8,8,8)
        upscaled_embedding: torch.Tensor,  # (B,32,128,128,128)
        prev_mask_prob: torch.Tensor,      # (B,1,128,128,128) float
        points=None,                       # (coords,labs)
        boxes=None,                        # (B,6) or None
        multimask_output: bool = True,
    ):
        device = image_embedding.device
        B = image_embedding.shape[0]

        # prev mask prompt
        masks_in = prev_mask_prob if prev_mask_prob is not None else None

        sparse, dense = self.prompt_encoder(points=points, boxes=boxes, masks=masks_in)

        # dense PE for the image tokens (8^3 => 512 tokens)
        image_pe = self.prompt_encoder.get_dense_pe(device=device).repeat(B, 1, 1)

        masks, scores = self.mask_decoder(
            image_embedding=image_embedding,
            image_pe=image_pe,
            sparse_prompt_embeddings=sparse,
            dense_prompt_embeddings=dense,
            upscaled_embedding=upscaled_embedding,
            multimask_output=multimask_output,
        )
        return masks, scores
''')

print('Wrote:', PKG_ROOT/'models'/'prism_sam3d.py')

Wrote: /content/prism3d_project/prism3d/models/prism_sam3d.py


In [81]:
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

DATA_ROOT = '/content/prism_data/colon'
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))

model = PrismSAM3D().to(device).eval()
runner = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=3,              # keep small for sanity
    num_clicks_train=10,      # keep small for sanity
    num_clicks_val=5,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

with torch.no_grad():
    out, final_logits = runner.run(img, lab, train=False)

print('val-style mean dice over iters:', float(out))
print('final_logits:', tuple(final_logits.shape))

val-style mean dice over iters: 0.15492868423461914
final_logits: (1, 1, 128, 128, 128)


In [82]:
from pathlib import Path
import textwrap
import importlib

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'prompt_encoder3d.py', '''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple


class PositionEmbeddingRandom3D(nn.Module):
    def __init__(self, num_pos_feats: int = 192, scale: float = 1.0):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.scale = scale
        self.register_buffer('gaussian_matrix', torch.randn(3, num_pos_feats) * scale)

    def _pe(self, coords: torch.Tensor) -> torch.Tensor:
        proj = 2 * math.pi * (coords @ self.gaussian_matrix)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

    def forward_with_coords(self, coords_zyx: torch.Tensor, image_size: int) -> torch.Tensor:
        coords = coords_zyx / float(image_size - 1)
        coords = coords.clamp(0.0, 1.0)
        return self._pe(coords)

    def forward_grid(self, grid_size: Tuple[int, int, int], device: torch.device) -> torch.Tensor:
        D, H, W = grid_size
        z = torch.linspace(0.0, 1.0, steps=D, device=device)
        y = torch.linspace(0.0, 1.0, steps=H, device=device)
        x = torch.linspace(0.0, 1.0, steps=W, device=device)
        zz, yy, xx = torch.meshgrid(z, y, x, indexing='ij')
        coords = torch.stack([zz, yy, xx], dim=-1).view(-1, 3)
        pe = self._pe(coords).unsqueeze(0)
        return pe


class MaskDownscaler3D(nn.Module):
    def __init__(self, embed_dim: int = 384):
        super().__init__()
        ch = 32
        self.net = nn.Sequential(
            nn.Conv3d(1, ch, kernel_size=2, stride=2),
            nn.InstanceNorm3d(ch, affine=True),
            nn.GELU(),

            nn.Conv3d(ch, ch, kernel_size=2, stride=2),
            nn.InstanceNorm3d(ch, affine=True),
            nn.GELU(),

            nn.Conv3d(ch, ch * 2, kernel_size=2, stride=2),
            nn.InstanceNorm3d(ch * 2, affine=True),
            nn.GELU(),

            nn.Conv3d(ch * 2, ch * 4, kernel_size=2, stride=2),
            nn.InstanceNorm3d(ch * 4, affine=True),
            nn.GELU(),

            nn.Conv3d(ch * 4, embed_dim, kernel_size=1),
        )

    def forward(self, mask: torch.Tensor) -> torch.Tensor:
        return self.net(mask)


class PromptEncoder3D(nn.Module):
    def __init__(self, embed_dim: int = 384, image_size: int = 128):
        super().__init__()
        self.embed_dim = embed_dim
        self.image_size = image_size

        self.pe_layer = PositionEmbeddingRandom3D(num_pos_feats=embed_dim // 2)

        self.pos_point_embed = nn.Embedding(1, embed_dim)
        self.neg_point_embed = nn.Embedding(1, embed_dim)
        self.not_a_point_embed = nn.Embedding(1, embed_dim)

        self.corner0_embed = nn.Embedding(1, embed_dim)
        self.corner1_embed = nn.Embedding(1, embed_dim)

        self.mask_downscaler = MaskDownscaler3D(embed_dim=embed_dim)
        self.no_mask_embed = nn.Embedding(1, embed_dim)

    def get_dense_pe(self, device: torch.device) -> torch.Tensor:
        return self.pe_layer.forward_grid((8, 8, 8), device=device)

    def forward(
        self,
        points: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        boxes: Optional[torch.Tensor] = None,
        masks: Optional[torch.Tensor] = None,
    ):
        device = None
        B = 1
        sparse_embeddings = []

        # Points
        if points is not None:
            coords, labels = points
            device = coords.device
            B = coords.shape[0]

            pe = self.pe_layer.forward_with_coords(coords, self.image_size)  # (B,N,384)
            out = pe.clone()

            # AMP-safe: match dtype to out
            pos = self.pos_point_embed.weight[0].to(dtype=out.dtype, device=out.device)
            neg = self.neg_point_embed.weight[0].to(dtype=out.dtype, device=out.device)
            pad = self.not_a_point_embed.weight[0].to(dtype=out.dtype, device=out.device)

            out[labels == 1] = out[labels == 1] + pos
            out[labels == 0] = out[labels == 0] + neg
            out[labels == -1] = out[labels == -1] + pad
            sparse_embeddings.append(out)

        # Boxes
        if boxes is not None:
            if boxes.dim() == 2 and boxes.shape[1] == 6:
                corner0 = boxes[:, 0:3]
                corner1 = boxes[:, 3:6]
            elif boxes.dim() == 3 and boxes.shape[1] == 2 and boxes.shape[2] == 3:
                corner0 = boxes[:, 0, :]
                corner1 = boxes[:, 1, :]
            else:
                raise ValueError('boxes must be (B,6) or (B,2,3)')

            device = boxes.device if device is None else device
            B = boxes.shape[0]

            c0_pe = self.pe_layer.forward_with_coords(corner0.unsqueeze(1), self.image_size)[:, 0, :]
            c1_pe = self.pe_layer.forward_with_coords(corner1.unsqueeze(1), self.image_size)[:, 0, :]

            # AMP-safe: match dtype
            e0 = self.corner0_embed.weight[0].to(dtype=c0_pe.dtype, device=c0_pe.device)
            e1 = self.corner1_embed.weight[0].to(dtype=c1_pe.dtype, device=c1_pe.device)

            c0 = c0_pe + e0
            c1 = c1_pe + e1
            sparse_embeddings.append(torch.stack([c0, c1], dim=1))

        if len(sparse_embeddings) == 0:
            if device is None:
                device = torch.device('cpu')
            sparse = torch.zeros((B, 0, self.embed_dim), device=device)
        else:
            sparse = torch.cat(sparse_embeddings, dim=1)

        # Dense mask prompt
        if masks is not None:
            dense = self.mask_downscaler(masks)
        else:
            if device is None:
                device = torch.device('cpu')
            base = self.no_mask_embed.weight[0].to(device=device)
            dense = base.view(1, self.embed_dim, 1, 1, 1).repeat(B, 1, 8, 8, 8)

        return sparse, dense
''')

importlib.invalidate_caches()
print('Patched:', PKG_ROOT/'models'/'prompt_encoder3d.py')

Patched: /content/prism3d_project/prism3d/models/prompt_encoder3d.py


In [83]:
import torch
import torchio as tio
from torch.utils.data import DataLoader
from torch.cuda import amp
import importlib

# reload modules so new class is used
import prism3d.models.prompt_encoder3d as pe_mod
import prism3d.models.prism_sam3d as prism_mod
importlib.reload(pe_mod)
importlib.reload(prism_mod)

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

DATA_ROOT = '/content/prism_data/colon'
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

model = PrismSAM3D().to(device).train()
runner = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=2,
    num_clicks_train=10,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

opt = torch.optim.AdamW(model.parameters(), lr=4e-5)
scaler = amp.GradScaler()

img, lab, _ = next(iter(dl))
img = img.to(device)
lab = lab.to(device)

opt.zero_grad(set_to_none=True)
with amp.autocast():
    loss, _ = runner.run(img, lab, train=True)

scaler.scale(loss).backward()
scaler.unscale_(opt)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(opt)
scaler.update()

print('train loss:', float(loss.detach().cpu()))

/tmp/ipython-input-3943267322.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
/tmp/ipython-input-3943267322.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


train loss: 4.8596720695495605


In [84]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Modern AMP API (no deprecation warnings)
def autocast_ctx():
    if device == 'cuda':
        return torch.amp.autocast('cuda')
    # CPU autocast exists but is not useful here; keep it off
    class _NullCtx:
        def __enter__(self): return None
        def __exit__(self, exc_type, exc, tb): return False
    return _NullCtx()

scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None

print('device:', device)
print('amp enabled:', device == 'cuda')

device: cuda
amp enabled: True


In [85]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data
train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

DATA_ROOT = '/content/prism_data/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
ds_val   = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='val',   torchio_transform=val_tio)

dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=1, shuffle=False, num_workers=0)

# Model
model = PrismSAM3D().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.01)

# Debug interactive settings (fast)
runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=2,
    num_clicks_train=10,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=2,
    num_clicks_val=5,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

# Checkpoint dir
CKPT_DIR = '/content/prism_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True)

def run_val(max_batches=2):
    model.eval()
    dices = []
    with torch.no_grad():
        for i, (img, lab, _) in enumerate(dl_val):
            if i >= max_batches:
                break
            img = img.to(device)
            lab = lab.to(device)
            d, _ = runner_val.run(img, lab, train=False)
            dices.append(float(d.detach().cpu()))
    return sum(dices) / max(1, len(dices))

# Train loop (debug)
epochs = 2
steps_per_epoch = 2  # keep small for debug
best_val = -1.0

for ep in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    losses = []

    for step, (img, lab, _) in enumerate(dl_train):
        if step >= steps_per_epoch:
            break

        img = img.to(device)
        lab = lab.to(device)

        opt.zero_grad(set_to_none=True)
        with autocast_ctx():
            loss, _ = runner_train.run(img, lab, train=True)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        losses.append(float(loss.detach().cpu()))

    val_dice = run_val(max_batches=1)
    mean_loss = sum(losses) / max(1, len(losses))
    dt = time.time() - t0

    print(f'epoch {ep} | loss {mean_loss:.4f} | val_dice {val_dice:.4f} | time {dt:.1f}s')

    # save last
    last_path = os.path.join(CKPT_DIR, 'last.pt')
    torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict()}, last_path)

    # save best
    if val_dice > best_val:
        best_val = val_dice
        best_path = os.path.join(CKPT_DIR, 'best.pt')
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(), 'best_val': best_val}, best_path)
        print('  saved best ->', best_path)

print('best_val:', best_val)
print('ckpt dir:', CKPT_DIR)

epoch 1 | loss 4.9215 | val_dice 0.0888 | time 2.3s
  saved best -> /content/prism_ckpts/best.pt
epoch 2 | loss 4.6323 | val_dice 0.0886 | time 1.9s
best_val: 0.08879300951957703
ckpt dir: /content/prism_ckpts


In [86]:
import os, torch
from prism3d.models.prism_sam3d import PrismSAM3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR = '/content/prism_ckpts'
ckpt_path = os.path.join(CKPT_DIR, 'best.pt')

model = PrismSAM3D().to(device).eval()
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model'], strict=True)

print('Loaded:', ckpt_path)
print('epoch:', ckpt.get('epoch'))
print('best_val:', ckpt.get('best_val'))

Loaded: /content/prism_ckpts/best.pt
epoch: 1
best_val: 0.08879300951957703


In [87]:
from pathlib import Path
import textwrap
import importlib

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_runner.py', '''
import torch
from prism3d.engine.interactive_utils import sample_click_points, bbox_from_mask_zyx, PrismLosses3D


class InteractiveRunner3D:
    def __init__(
        self,
        model,
        device: str = 'cuda',
        iter_nums: int = 11,
        num_clicks_train: int = 50,
        num_clicks_val: int = 10,
        dynamic_clicks: bool = True,
        use_box: bool = True,
        dynamic_box: bool = False,
        boundary_kernel_size: int = 5,
        multiple_outputs: bool = True,
    ):
        self.model = model
        self.device = device

        self.iter_nums = iter_nums
        self.num_clicks_train = num_clicks_train
        self.num_clicks_val = num_clicks_val
        self.dynamic_clicks = dynamic_clicks

        self.use_box = use_box
        self.dynamic_box = dynamic_box
        self.multiple_outputs = multiple_outputs

        self.losses = PrismLosses3D(boundary_kernel_size=boundary_kernel_size, device=device)

        self.click_points_hist = []
        self.click_labels_hist = []

    def run(self, image: torch.Tensor, label: torch.Tensor, train: bool = True, return_curve: bool = False):
        image = image.to(self.device)
        label = label.to(self.device)

        self.click_points_hist = []
        self.click_labels_hist = []
        dice_curve = []

        image_emb, up_emb, _ = self.model.image_encoder(image)

        prev_logits = torch.zeros_like(label, dtype=torch.float32, device=self.device)

        total = 0.0
        for it in range(self.iter_nums):
            prev_prob = torch.sigmoid(prev_logits) if it > 0 else prev_logits

            mode = 'train' if train else 'validation'
            num_clicks = self.num_clicks_train if train else self.num_clicks_val

            coords, labs = sample_click_points(prev_prob, label, num_clicks=num_clicks, dynamic=self.dynamic_clicks, mode=mode)

            self.click_points_hist.append(coords)
            self.click_labels_hist.append(labs)

            box = None
            if self.use_box:
                gt = (label[:, 0] > 0)
                box = bbox_from_mask_zyx(gt, mode=mode, dynamic=self.dynamic_box).float()

            masks, scores = self.model.step(
                image_embedding=image_emb,
                upscaled_embedding=up_emb,
                prev_mask_prob=prev_prob,
                points=(coords, labs),
                boxes=box,
                multimask_output=True,
            )

            if self.multiple_outputs:
                best_idx = torch.argmax(scores, dim=1)
                best = torch.stack([masks[b, best_idx[b]] for b in range(masks.shape[0])], dim=0).unsqueeze(1)
            else:
                best = masks[:, 0:1]

            # dice for curve (always)
            prob = torch.sigmoid(best)
            pred = (prob > 0.5).float()
            gt = (label > 0).float()
            inter = (pred * gt).sum(dim=(1,2,3,4))
            denom = pred.sum(dim=(1,2,3,4)) + gt.sum(dim=(1,2,3,4)) + 1e-8
            dice = (2.0 * inter / denom).mean()
            dice_curve.append(dice)

            if train:
                loss = 0.0
                for k in range(masks.shape[1]):
                    mk = masks[:, k:k+1]
                    sk = scores[:, k]
                    loss = loss + self.losses(mk, label, sk)
                total = total + loss
            else:
                total = total + dice

            prev_logits = best

        mean = total / float(self.iter_nums)
        if return_curve:
            return mean, prev_logits, dice_curve
        return mean, prev_logits
''')

importlib.invalidate_caches()
import prism3d.engine.interactive_runner as ir
importlib.reload(ir)
print('Patched runner with dice curves.')

Patched runner with dice curves.


In [88]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

DATA_ROOT = '/content/prism_data/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
ds_val   = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='val',   torchio_transform=val_tio)

dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=1, shuffle=False, num_workers=0)

model = PrismSAM3D().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.01)

# PAPER-LIKE settings
runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_train=50,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_val=10,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
)

CKPT_DIR = '/content/prism_ckpts_paper'
os.makedirs(CKPT_DIR, exist_ok=True)

def validate_full(max_batches=None):
    model.eval()
    dice_list = []
    curve_sum = None
    n = 0

    with torch.no_grad():
        for i, (img, lab, _) in enumerate(dl_val):
            if max_batches is not None and i >= max_batches:
                break
            img = img.to(device)
            lab = lab.to(device)

            mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
            dice_list.append(float(mean_dice.detach().cpu()))

            curve_vals = [float(c.detach().cpu()) for c in curve]
            if curve_sum is None:
                curve_sum = curve_vals
            else:
                curve_sum = [a + b for a, b in zip(curve_sum, curve_vals)]
            n += 1

    mean_dice = sum(dice_list) / max(1, len(dice_list))
    mean_curve = [c / max(1, n) for c in curve_sum] if curve_sum is not None else []
    return mean_dice, mean_curve

# For speed on Colab, start with a tiny run: 1 epoch, few steps
epochs = 1
steps_per_epoch = 2   # raise later
best_val = -1.0

for ep in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    losses = []

    for step, (img, lab, _) in enumerate(dl_train):
        if step >= steps_per_epoch:
            break
        img = img.to(device)
        lab = lab.to(device)

        opt.zero_grad(set_to_none=True)
        with autocast_ctx():
            loss, _ = runner_train.run(img, lab, train=True)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        losses.append(float(loss.detach().cpu()))

    # full val (for your synthetic dataset it is tiny anyway)
    val_dice, val_curve = validate_full(max_batches=None)

    mean_loss = sum(losses) / max(1, len(losses))
    dt = time.time() - t0

    print(f'epoch {ep} | loss {mean_loss:.4f} | val_dice {val_dice:.4f} | time {dt:.1f}s')
    print('val dice curve per iter:')
    for i, d in enumerate(val_curve, start=1):
        print(f'  iter {i:02d}: {d:.4f}')

    # save
    torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict()}, os.path.join(CKPT_DIR, 'last.pt'))
    if val_dice > best_val:
        best_val = val_dice
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(), 'best_val': best_val},
                   os.path.join(CKPT_DIR, 'best.pt'))
        print('saved best')

print('best_val:', best_val)
print('ckpt dir:', CKPT_DIR)

epoch 1 | loss 4.9628 | val_dice 0.0769 | time 4.3s
val dice curve per iter:
  iter 01: 0.0793
  iter 02: 0.0772
  iter 03: 0.0765
  iter 04: 0.0773
  iter 05: 0.0760
  iter 06: 0.0756
  iter 07: 0.0770
  iter 08: 0.0763
  iter 09: 0.0772
  iter 10: 0.0766
  iter 11: 0.0769
saved best
best_val: 0.07688753306865692
ckpt dir: /content/prism_ckpts_paper


In [89]:
from pathlib import Path
import textwrap
import importlib

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_runner.py', '''
import torch
from prism3d.engine.interactive_utils import sample_click_points, bbox_from_mask_zyx, PrismLosses3D


class InteractiveRunner3D:
    def __init__(
        self,
        model,
        device: str = 'cuda',
        iter_nums: int = 11,
        num_clicks_train: int = 50,
        num_clicks_val: int = 10,
        dynamic_clicks: bool = True,
        use_box: bool = True,
        dynamic_box: bool = False,
        boundary_kernel_size: int = 5,
        multiple_outputs: bool = True,
        detach_prev_mask: bool = True,
        freeze_image_encoder: bool = True,
    ):
        self.model = model
        self.device = device

        self.iter_nums = iter_nums
        self.num_clicks_train = num_clicks_train
        self.num_clicks_val = num_clicks_val
        self.dynamic_clicks = dynamic_clicks

        self.use_box = use_box
        self.dynamic_box = dynamic_box
        self.multiple_outputs = multiple_outputs

        self.detach_prev_mask = detach_prev_mask
        self.freeze_image_encoder = freeze_image_encoder

        self.losses = PrismLosses3D(boundary_kernel_size=boundary_kernel_size, device=device)

        self.click_points_hist = []
        self.click_labels_hist = []

        if self.freeze_image_encoder:
            for p in self.model.image_encoder.parameters():
                p.requires_grad_(False)

    def _select_best(self, masks: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        if self.multiple_outputs:
            best_idx = torch.argmax(scores, dim=1)
            best = torch.stack([masks[b, best_idx[b]] for b in range(masks.shape[0])], dim=0).unsqueeze(1)
            return best
        return masks[:, 0:1]

    def _dice(self, best_logits: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        prob = torch.sigmoid(best_logits)
        pred = (prob > 0.5).float()
        gt = (label > 0).float()
        inter = (pred * gt).sum(dim=(1,2,3,4))
        denom = pred.sum(dim=(1,2,3,4)) + gt.sum(dim=(1,2,3,4)) + 1e-8
        return (2.0 * inter / denom).mean()

    def run(self, image: torch.Tensor, label: torch.Tensor, train: bool = True, return_curve: bool = False):
        image = image.to(self.device)
        label = label.to(self.device)

        self.click_points_hist = []
        self.click_labels_hist = []
        dice_curve = [] if return_curve else None

        # image features
        if self.freeze_image_encoder:
            with torch.no_grad():
                image_emb, up_emb, _ = self.model.image_encoder(image)
        else:
            image_emb, up_emb, _ = self.model.image_encoder(image)

        prev_logits = torch.zeros_like(label, dtype=torch.float32, device=self.device)
        total = 0.0

        for it in range(self.iter_nums):
            prev_prob = torch.sigmoid(prev_logits) if it > 0 else prev_logits
            if self.detach_prev_mask:
                prev_prob = prev_prob.detach()

            mode = 'train' if train else 'validation'
            num_clicks = self.num_clicks_train if train else self.num_clicks_val
            coords, labs = sample_click_points(prev_prob, label, num_clicks=num_clicks, dynamic=self.dynamic_clicks, mode=mode)

            self.click_points_hist.append(coords)
            self.click_labels_hist.append(labs)

            box = None
            if self.use_box:
                gt = (label[:, 0] > 0)
                box = bbox_from_mask_zyx(gt, mode=mode, dynamic=self.dynamic_box).float()

            masks, scores = self.model.step(
                image_embedding=image_emb,
                upscaled_embedding=up_emb,
                prev_mask_prob=prev_prob,
                points=(coords, labs),
                boxes=box,
                multimask_output=True,
            )

            best = self._select_best(masks, scores)

            if not train:
                d = self._dice(best, label)
                total = total + d
                if return_curve:
                    dice_curve.append(d)
            else:
                # if someone uses run(train=True) for debugging without backward:
                loss = 0.0
                for k in range(masks.shape[1]):
                    loss = loss + self.losses(masks[:, k:k+1], label, scores[:, k])
                total = total + loss

            prev_logits = best.detach() if self.detach_prev_mask else best

        mean = total / float(self.iter_nums)
        if return_curve:
            return mean, prev_logits, dice_curve
        return mean, prev_logits

    def train_step(self, image: torch.Tensor, label: torch.Tensor, opt, scaler, autocast_ctx, grad_clip: float = 1.0):
        image = image.to(self.device)
        label = label.to(self.device)

        self.click_points_hist = []
        self.click_labels_hist = []

        # compute image features once
        if self.freeze_image_encoder:
            with torch.no_grad():
                image_emb, up_emb, _ = self.model.image_encoder(image)
        else:
            # if you unfreeze encoder, this will be heavier
            image_emb, up_emb, _ = self.model.image_encoder(image)

        prev_logits = torch.zeros_like(label, dtype=torch.float32, device=self.device)

        opt.zero_grad(set_to_none=True)
        loss_sum = 0.0

        for it in range(self.iter_nums):
            prev_prob = torch.sigmoid(prev_logits) if it > 0 else prev_logits
            if self.detach_prev_mask:
                prev_prob = prev_prob.detach()

            mode = 'train'
            coords, labs = sample_click_points(prev_prob, label, num_clicks=self.num_clicks_train, dynamic=self.dynamic_clicks, mode=mode)

            self.click_points_hist.append(coords)
            self.click_labels_hist.append(labs)

            box = None
            if self.use_box:
                gt = (label[:, 0] > 0)
                box = bbox_from_mask_zyx(gt, mode=mode, dynamic=self.dynamic_box).float()

            with autocast_ctx():
                masks, scores = self.model.step(
                    image_embedding=image_emb,
                    upscaled_embedding=up_emb,
                    prev_mask_prob=prev_prob,
                    points=(coords, labs),
                    boxes=box,
                    multimask_output=True,
                )

                loss_it = 0.0
                for k in range(masks.shape[1]):
                    loss_it = loss_it + self.losses(masks[:, k:k+1], label, scores[:, k])

                # scale per-iter so total magnitude stays similar
                loss_scaled = loss_it / float(self.iter_nums)

            if scaler is not None:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            loss_sum = loss_sum + float(loss_it.detach().cpu())

            best = self._select_best(masks, scores)
            prev_logits = best.detach() if self.detach_prev_mask else best

        if scaler is not None:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
            opt.step()

        mean_loss = loss_sum / float(self.iter_nums)
        return mean_loss, prev_logits
''')

importlib.invalidate_caches()
import prism3d.engine.interactive_runner as ir
importlib.reload(ir)
print('Patched runner: memory-safe train_step + no dice curve in train.')

Patched runner: memory-safe train_step + no dice curve in train.


In [90]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

DATA_ROOT = '/content/prism_data/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
ds_val   = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='val',   torchio_transform=val_tio)

dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=1, shuffle=False, num_workers=0)

model = PrismSAM3D().to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-5, weight_decay=0.01)

runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_train=50,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,   # key for memory
)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_val=10,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,
)

CKPT_DIR = '/content/prism_ckpts_paper'
os.makedirs(CKPT_DIR, exist_ok=True)

def validate_full():
    model.eval()
    dice_list = []
    curve_sum = None
    n = 0
    with torch.no_grad():
        for img, lab, _ in dl_val:
            img = img.to(device)
            lab = lab.to(device)
            mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
            dice_list.append(float(mean_dice.detach().cpu()))
            curve_vals = [float(c.detach().cpu()) for c in curve]
            curve_sum = curve_vals if curve_sum is None else [a + b for a, b in zip(curve_sum, curve_vals)]
            n += 1
    mean_dice = sum(dice_list) / max(1, len(dice_list))
    mean_curve = [c / max(1, n) for c in curve_sum]
    return mean_dice, mean_curve

epochs = 1
steps_per_epoch = 2
best_val = -1.0

for ep in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    losses = []

    for step, (img, lab, _) in enumerate(dl_train):
        if step >= steps_per_epoch:
            break
        mean_loss, _ = runner_train.train_step(img, lab, opt=opt, scaler=scaler, autocast_ctx=autocast_ctx, grad_clip=1.0)
        losses.append(mean_loss)

    val_dice, val_curve = validate_full()
    mean_loss_epoch = sum(losses) / max(1, len(losses))
    dt = time.time() - t0

    print(f'epoch {ep} | loss {mean_loss_epoch:.4f} | val_dice {val_dice:.4f} | time {dt:.1f}s')
    print('val dice curve per iter:')
    for i, d in enumerate(val_curve, start=1):
        print(f'  iter {i:02d}: {d:.4f}')

    torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict()}, os.path.join(CKPT_DIR, 'last.pt'))
    if val_dice > best_val:
        best_val = val_dice
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(), 'best_val': best_val},
                   os.path.join(CKPT_DIR, 'best.pt'))
        print('saved best')

print('best_val:', best_val)
print('ckpt dir:', CKPT_DIR)

epoch 1 | loss 4.8820 | val_dice 0.0818 | time 4.0s
val dice curve per iter:
  iter 01: 0.0922
  iter 02: 0.0809
  iter 03: 0.0805
  iter 04: 0.0805
  iter 05: 0.0810
  iter 06: 0.0800
  iter 07: 0.0810
  iter 08: 0.0813
  iter 09: 0.0821
  iter 10: 0.0803
  iter 11: 0.0802
saved best
best_val: 0.08180573582649231
ckpt dir: /content/prism_ckpts_paper


In [91]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4983 MiB |  11406 MiB |   1036 GiB |   1032 GiB |
|       from large pool |   4715 MiB |  11398 MiB |   1017 GiB |   1012 GiB |
|       from small pool |    268 MiB |    473 MiB |     19 GiB |     19 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4983 MiB |  11406 MiB |   1036 GiB |   1032 GiB |
|       from large pool |   4715 MiB |  11398 MiB |   1017 GiB |

In [92]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# BIG FIX: prevent CuDNN from choosing huge-workspace conv algorithms
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# optional: allow TF32 for matmul (not a memory saver, but fine)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print('cudnn.benchmark:', torch.backends.cudnn.benchmark)
print('cudnn.deterministic:', torch.backends.cudnn.deterministic)

cudnn.benchmark: False
cudnn.deterministic: True


In [93]:
from pathlib import Path
import textwrap
import importlib

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_utils.py', '''
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.losses import DiceCELoss, DiceLoss


def bbox_from_mask_zyx(mask_volume: torch.Tensor, mode: str = 'train', dynamic: bool = False, max_diff: int = 10) -> torch.Tensor:
    bboxes = []
    for vol in mask_volume:
        i_any = vol.any(dim=2).any(dim=1)
        j_any = vol.any(dim=2).any(dim=0)
        k_any = vol.any(dim=1).any(dim=0)

        i_min, i_max = torch.where(i_any)[0][[0, -1]]
        j_min, j_max = torch.where(j_any)[0][[0, -1]]
        k_min, k_max = torch.where(k_any)[0][[0, -1]]

        if dynamic and mode == 'train':
            diff_ = torch.randint(low=0, high=max_diff, size=(6,), device=vol.device)
            if max(0, int(i_min) - int(diff_[0])) < min(int(i_max) + int(diff_[1]), 126):
                i_min = max(0, int(i_min) - int(diff_[0]))
                i_max = min(int(i_max) + int(diff_[1]), 126)
            if max(0, int(j_min) - int(diff_[2])) < min(int(j_max) + int(diff_[3]), 126):
                j_min = max(0, int(j_min) - int(diff_[2]))
                j_max = min(int(j_max) + int(diff_[3]), 126)
            if max(0, int(k_min) - int(diff_[4])) < min(int(k_max) + int(diff_[5]), 126):
                k_min = max(0, int(k_min) - int(diff_[4]))
                k_max = min(int(k_max) + int(diff_[5]), 126)

        bb = torch.tensor([int(i_min), int(j_min), int(k_min), int(i_max) + 1, int(j_max) + 1, int(k_max) + 1], device=vol.device)
        bboxes.append(bb)

    return torch.stack(bboxes, dim=0)


def sample_click_points(prev_prob: torch.Tensor, label: torch.Tensor, num_clicks: int, dynamic: bool, mode: str):
    device = label.device
    B = label.shape[0]

    pred_masks = (prev_prob > 0.5)
    true_masks = (label > 0)

    fn_masks = torch.logical_and(true_masks, torch.logical_not(pred_masks))
    fp_masks = torch.logical_and(torch.logical_not(true_masks), pred_masks)
    to_point_mask = torch.logical_or(fn_masks, fp_masks)  # (B,1,D,H,W)

    # move to CPU for indexing
    to_point_cpu = to_point_mask[:, 0].detach().to('cpu')
    fn_cpu = fn_masks[:, 0].detach().to('cpu')

    points_list = [int(torch.count_nonzero(to_point_cpu[i]).item()) for i in range(B)]
    points_min = min(points_list) if len(points_list) > 0 else 0

    if points_min == 0:
        coords = torch.zeros((B, 0, 3), device=device).float()
        labs = torch.zeros((B, 0), device=device).long()
        return coords, labs

    click_size = points_min if num_clicks > points_min else num_clicks
    if dynamic and mode == 'train':
        dynamic_size = random.randint(1, max(1, int(click_size)))
    else:
        dynamic_size = int(click_size)

    batch_coords = []
    batch_labs = []

    for i in range(B):
        pts = torch.nonzero(to_point_cpu[i], as_tuple=False)  # (N,3) in z,y,x
        perm = torch.randperm(pts.shape[0])[:dynamic_size]
        pts_sel = pts[perm]

        coords_i = []
        labs_i = []
        for zyx in pts_sel:
            z, y, x = int(zyx[0]), int(zyx[1]), int(zyx[2])
            is_pos = bool(fn_cpu[i, z, y, x].item())
            coords_i.append(torch.tensor([[z, y, x]], dtype=torch.float32))
            labs_i.append(torch.tensor([1 if is_pos else 0], dtype=torch.long))

        coords_i = torch.cat(coords_i, dim=0).unsqueeze(0)  # (1,N,3)
        labs_i = torch.cat(labs_i, dim=0).unsqueeze(0)      # (1,N)

        batch_coords.append(coords_i)
        batch_labs.append(labs_i)

    coords = torch.cat(batch_coords, dim=0).to(device=device)
    labs = torch.cat(batch_labs, dim=0).to(device=device)
    return coords, labs


class PrismLosses3D(nn.Module):
    def __init__(self, boundary_kernel_size: int = 5, device: str = 'cuda'):
        super().__init__()
        pad = int((boundary_kernel_size - 1) / 2)
        self.pool = nn.AvgPool3d((boundary_kernel_size, boundary_kernel_size, 1), stride=1, padding=(pad, pad, 0)).to(device)

        self.loss_boundary = nn.MSELoss()
        self.loss_seg = DiceCELoss(sigmoid=True, squared_pred=True, reduction='mean')
        self.loss_val_dice = DiceLoss(sigmoid=True, reduction='none')

    def forward(self, mask_logits: torch.Tensor, label: torch.Tensor, pred_score: torch.Tensor) -> torch.Tensor:
        mask_prob = torch.sigmoid(mask_logits)

        seg_edge = (label - self.pool(label)).abs()
        mask_edge = (mask_prob - self.pool(mask_prob)).abs()

        pred_dice_score_loss = 0.0
        for b in range(mask_logits.shape[0]):
            dl = self.loss_val_dice(mask_logits[b:b+1], label[b:b+1])
            target_dice = 1.0 - dl.mean()
            pred_dice_score_loss = pred_dice_score_loss + self.loss_boundary(pred_score[b], target_dice)

        loss = self.loss_seg(mask_logits, label) + 10.0 * self.loss_boundary(mask_edge, seg_edge)
        loss = loss + pred_dice_score_loss
        return loss
''')

importlib.invalidate_caches()
import prism3d.engine.interactive_utils as iu
importlib.reload(iu)
print('Patched interactive_utils: CPU click sampling.')

Patched interactive_utils: CPU click sampling.


In [94]:
import gc
import importlib
import torch

import prism3d.engine.interactive_runner as ir
importlib.reload(ir)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Reloaded runner + cleared cache.')

Reloaded runner + cleared cache.


In [95]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

DATA_ROOT = '/content/prism_data/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
ds_val   = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='val',   torchio_transform=val_tio)

dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=1, shuffle=False, num_workers=0)

model = PrismSAM3D().to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-5, weight_decay=0.01)

runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_train=50,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,   # key for memory
)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_val=10,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,
)

CKPT_DIR = '/content/prism_ckpts_paper'
os.makedirs(CKPT_DIR, exist_ok=True)

def validate_full():
    model.eval()
    dice_list = []
    curve_sum = None
    n = 0
    with torch.no_grad():
        for img, lab, _ in dl_val:
            img = img.to(device)
            lab = lab.to(device)
            mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
            dice_list.append(float(mean_dice.detach().cpu()))
            curve_vals = [float(c.detach().cpu()) for c in curve]
            curve_sum = curve_vals if curve_sum is None else [a + b for a, b in zip(curve_sum, curve_vals)]
            n += 1
    mean_dice = sum(dice_list) / max(1, len(dice_list))
    mean_curve = [c / max(1, n) for c in curve_sum]
    return mean_dice, mean_curve

epochs = 1
steps_per_epoch = 2
best_val = -1.0

for ep in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    losses = []

    for step, (img, lab, _) in enumerate(dl_train):
        if step >= steps_per_epoch:
            break
        mean_loss, _ = runner_train.train_step(img, lab, opt=opt, scaler=scaler, autocast_ctx=autocast_ctx, grad_clip=1.0)
        losses.append(mean_loss)

    val_dice, val_curve = validate_full()
    mean_loss_epoch = sum(losses) / max(1, len(losses))
    dt = time.time() - t0

    print(f'epoch {ep} | loss {mean_loss_epoch:.4f} | val_dice {val_dice:.4f} | time {dt:.1f}s')
    print('val dice curve per iter:')
    for i, d in enumerate(val_curve, start=1):
        print(f'  iter {i:02d}: {d:.4f}')

    torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict()}, os.path.join(CKPT_DIR, 'last.pt'))
    if val_dice > best_val:
        best_val = val_dice
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(), 'best_val': best_val},
                   os.path.join(CKPT_DIR, 'best.pt'))
        print('saved best')

print('best_val:', best_val)
print('ckpt dir:', CKPT_DIR)

epoch 1 | loss 4.6484 | val_dice 0.0739 | time 3.8s
val dice curve per iter:
  iter 01: 0.0739
  iter 02: 0.0727
  iter 03: 0.0737
  iter 04: 0.0740
  iter 05: 0.0731
  iter 06: 0.0733
  iter 07: 0.0746
  iter 08: 0.0745
  iter 09: 0.0751
  iter 10: 0.0733
  iter 11: 0.0746
saved best
best_val: 0.0738980770111084
ckpt dir: /content/prism_ckpts_paper


In [96]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'train_utils.py', '''
import os
from dataclasses import dataclass
from typing import Optional, Dict, Any, List

import torch
from torch.utils.tensorboard import SummaryWriter


@dataclass
class TrainConfig:
    dataset: str = 'colon'
    data_root: str = '/content/prism_data/colon'
    epochs: int = 50
    steps_per_epoch: Optional[int] = None  # None => full epoch
    lr: float = 4e-5
    weight_decay: float = 0.01
    ckpt_dir: str = '/content/prism_ckpts_paper'
    log_dir: str = '/content/prism_tb'

    # interactive paper settings
    iter_nums: int = 11
    num_clicks_train: int = 50
    num_clicks_val: int = 10

    # memory + stability
    freeze_image_encoder: bool = True
    detach_prev_mask: bool = True
    grad_clip: float = 1.0


class TBLogger:
    def __init__(self, log_dir: str):
        os.makedirs(log_dir, exist_ok=True)
        self.w = SummaryWriter(log_dir=log_dir)

    def log_scalars(self, tag_to_val: Dict[str, float], step: int):
        for k, v in tag_to_val.items():
            self.w.add_scalar(k, v, step)

    def log_curve(self, tag: str, curve: List[float], step: int):
        for i, v in enumerate(curve, start=1):
            self.w.add_scalar(f'{tag}/iter_{i:02d}', v, step)

    def close(self):
        self.w.flush()
        self.w.close()


def save_ckpt(path: str, payload: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def load_ckpt(path: str, device: str):
    return torch.load(path, map_location=device)
''')

print('Wrote:', PKG_ROOT/'engine'/'train_utils.py')

Wrote: /content/prism3d_project/prism3d/engine/train_utils.py


In [97]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D
from prism3d.engine.train_utils import TrainConfig, TBLogger, save_ckpt, load_ckpt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = TrainConfig(
    dataset='colon',
    data_root='/content/prism_data/colon',
    epochs=5,                 # change to paper run later (e.g., 50)
    steps_per_epoch=10,       # None for full epoch; keep small for now
    ckpt_dir='/content/prism_ckpts_paper',
    log_dir='/content/prism_tb',
)

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

ds_train = PrismCTDataset(dataset=cfg.dataset, data_dir=cfg.data_root, split='train', torchio_transform=train_tio)
ds_val   = PrismCTDataset(dataset=cfg.dataset, data_dir=cfg.data_root, split='val',   torchio_transform=val_tio)

dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=1, shuffle=False, num_workers=0)

model = PrismSAM3D().to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)

runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=cfg.iter_nums,
    num_clicks_train=cfg.num_clicks_train,
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=cfg.detach_prev_mask,
    freeze_image_encoder=cfg.freeze_image_encoder,
)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=cfg.iter_nums,
    num_clicks_val=cfg.num_clicks_val,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=cfg.detach_prev_mask,
    freeze_image_encoder=cfg.freeze_image_encoder,
)

def validate_full():
    model.eval()
    dice_list = []
    curve_sum = None
    n = 0
    with torch.no_grad():
        for img, lab, _ in dl_val:
            img = img.to(device)
            lab = lab.to(device)
            mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
            dice_list.append(float(mean_dice.detach().cpu()))
            curve_vals = [float(c.detach().cpu()) for c in curve]
            curve_sum = curve_vals if curve_sum is None else [a + b for a, b in zip(curve_sum, curve_vals)]
            n += 1
    mean_dice = sum(dice_list) / max(1, len(dice_list))
    mean_curve = [c / max(1, n) for c in curve_sum] if curve_sum is not None else []
    return mean_dice, mean_curve

# Resume if exists
start_epoch = 1
best_val = -1.0
last_path = os.path.join(cfg.ckpt_dir, 'last.pt')
if os.path.exists(last_path):
    ckpt = load_ckpt(last_path, device=device)
    model.load_state_dict(ckpt['model'], strict=True)
    opt.load_state_dict(ckpt['opt'])
    start_epoch = int(ckpt.get('epoch', 0)) + 1
    best_val = float(ckpt.get('best_val', -1.0))
    print('Resumed from:', last_path, '| start_epoch:', start_epoch, '| best_val:', best_val)

tb = TBLogger(cfg.log_dir)
global_step = 0

for ep in range(start_epoch, cfg.epochs + 1):
    model.train()
    t0 = time.time()
    losses = []

    for step, (img, lab, _) in enumerate(dl_train):
        if cfg.steps_per_epoch is not None and step >= cfg.steps_per_epoch:
            break
        mean_loss, _ = runner_train.train_step(
            img, lab,
            opt=opt,
            scaler=scaler,
            autocast_ctx=autocast_ctx,
            grad_clip=cfg.grad_clip,
        )
        losses.append(mean_loss)
        tb.log_scalars({'train/loss_iter_mean': mean_loss}, global_step)
        global_step += 1

    val_dice, val_curve = validate_full()
    mean_loss_epoch = sum(losses) / max(1, len(losses))
    dt = time.time() - t0

    print(f'epoch {ep} | loss {mean_loss_epoch:.4f} | val_dice {val_dice:.4f} | time {dt:.1f}s')

    tb.log_scalars({'train/loss_epoch': mean_loss_epoch, 'val/dice': val_dice}, ep)
    tb.log_curve('val/dice_curve', val_curve, ep)

    payload = {
        'epoch': ep,
        'model': model.state_dict(),
        'opt': opt.state_dict(),
        'best_val': best_val,
        'cfg': cfg.__dict__,
    }
    save_ckpt(os.path.join(cfg.ckpt_dir, 'last.pt'), payload)

    if val_dice > best_val:
        best_val = val_dice
        payload['best_val'] = best_val
        save_ckpt(os.path.join(cfg.ckpt_dir, 'best.pt'), payload)
        print('  saved best')

tb.close()
print('done | best_val:', best_val)
print('ckpt:', cfg.ckpt_dir)
print('tb  :', cfg.log_dir)

Resumed from: /content/prism_ckpts_paper/last.pt | start_epoch: 2 | best_val: -1.0
epoch 2 | loss 4.7924 | val_dice 0.0738 | time 4.4s
  saved best
epoch 3 | loss 4.5600 | val_dice 0.0740 | time 3.8s
  saved best
epoch 4 | loss 4.6687 | val_dice 0.0663 | time 3.7s
epoch 5 | loss 4.5948 | val_dice 0.0574 | time 3.7s
done | best_val: 0.0740123987197876
ckpt: /content/prism_ckpts_paper
tb  : /content/prism_tb


In [98]:
%load_ext tensorboard
%tensorboard --logdir /content/prism_tb

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 1603), started 0:10:15 ago. (Use '!kill 1603' to kill it.)

<IPython.core.display.Javascript object>

In [99]:
import os
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D
from prism3d.engine.train_utils import load_ckpt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

CKPT_DIR = '/content/prism_ckpts_paper'
best_path = os.path.join(CKPT_DIR, 'best.pt')

ckpt = load_ckpt(best_path, device=device)

model = PrismSAM3D().to(device).eval()
model.load_state_dict(ckpt['model'], strict=True)

cfg = ckpt.get('cfg', {})
iter_nums = int(cfg.get('iter_nums', 11))
num_clicks_val = int(cfg.get('num_clicks_val', 10))

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

DATA_ROOT = cfg.get('data_root', '/content/prism_data/colon')
ds_val = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='val', torchio_transform=val_tio)
dl_val = DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=0)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=iter_nums,
    num_clicks_val=num_clicks_val,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,
)

dice_list = []
curve_sum = None
n = 0

with torch.no_grad():
    for img, lab, _ in dl_val:
        img = img.to(device)
        lab = lab.to(device)
        mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
        dice_list.append(float(mean_dice.detach().cpu()))
        curve_vals = [float(c.detach().cpu()) for c in curve]
        curve_sum = curve_vals if curve_sum is None else [a + b for a, b in zip(curve_sum, curve_vals)]
        n += 1

mean_dice = sum(dice_list) / max(1, len(dice_list))
mean_curve = [c / max(1, n) for c in curve_sum]

print('Loaded:', best_path)
print('Mean val dice:', mean_dice)
print('Mean dice curve:')
for i, d in enumerate(mean_curve, start=1):
    print(f'  iter {i:02d}: {d:.4f}')

Loaded: /content/prism_ckpts_paper/best.pt
Mean val dice: 0.07381528615951538
Mean dice curve:
  iter 01: 0.0749
  iter 02: 0.0735
  iter 03: 0.0733
  iter 04: 0.0732
  iter 05: 0.0744
  iter 06: 0.0730
  iter 07: 0.0738
  iter 08: 0.0738
  iter 09: 0.0744
  iter 10: 0.0741
  iter 11: 0.0736


In [100]:
!pip -q install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [101]:
import io
from pathlib import Path
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive = build('drive', 'v3')

ROOT_FOLDER_ID = '1bt17794HCZfmJ2MLh5w0Y_IAJyUj6ti2'
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_children(parent_id: str):
    page_token = None
    out = []
    while True:
        res = drive.files().list(
            q=f'''('{parent_id}' in parents) and trashed=false''',
            fields='nextPageToken, files(id, name, mimeType, size)',
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        out.extend(res.get('files', []))
        page_token = res.get('nextPageToken')
        if not page_token:
            break
    return out

top = list_children(ROOT_FOLDER_ID)
name_to_id = {c['name']: c['id'] for c in top}

imagesTr_id = name_to_id.get('imagesTr')
labelsTr_id = name_to_id.get('labelsTr')
split_id    = name_to_id.get('split.pkl')

if imagesTr_id is None or labelsTr_id is None:
    raise RuntimeError('Could not find imagesTr/labelsTr inside the provided Drive folder.')

print('imagesTr_id:', imagesTr_id)
print('labelsTr_id:', labelsTr_id)
print('split.pkl id:', split_id)

imagesTr_id: 1jMvQ3G2FsqSSsHDRZjjKqOZoPTnOM6tq
labelsTr_id: 1nEycgD5aW0Um80txa2xj4O7RnoGrLoTO
split.pkl id: 169bneBYZejTYKmbgogrhVqVUv5nhLyTu


In [102]:
from pathlib import Path

DEST = Path('/content/prism_data_tiny/colon')
(DEST/'imagesTr').mkdir(parents=True, exist_ok=True)
(DEST/'labelsTr').mkdir(parents=True, exist_ok=True)

def list_files(folder_id: str):
    page_token = None
    out = []
    while True:
        res = drive.files().list(
            q=f'''('{folder_id}' in parents) and trashed=false''',
            fields='nextPageToken, files(id, name, mimeType, size)',
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        out.extend(res.get('files', []))
        page_token = res.get('nextPageToken')
        if not page_token:
            break
    return out

def download_file(file_id: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print('SKIP:', dest.name)
        return
    request = drive.files().get_media(fileId=file_id)
    with io.FileIO(dest, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 8)
        done = False
        while not done:
            _, done = downloader.next_chunk()

imgs = [f for f in list_files(imagesTr_id) if f['name'].endswith('.nii.gz') or f['name'].endswith('.nii')]
labs = [f for f in list_files(labelsTr_id) if f['name'].endswith('.nii.gz') or f['name'].endswith('.nii')]

imgs = sorted(imgs, key=lambda x: x['name'])
lab_map = {f['name']: f for f in labs}

N = 5
picked = []
for f in imgs:
    if f['name'] in lab_map:
        picked.append(f)
    if len(picked) == N:
        break

if len(picked) < N:
    raise RuntimeError(f'Could only find {len(picked)} matching image/label pairs.')

print('Downloading', N, 'pairs to:', DEST)

for i, img in enumerate(picked, start=1):
    name = img['name']
    print(f'[{i}/{N}] image  -> {name}')
    download_file(img['id'], DEST/'imagesTr'/name)
    print(f'[{i}/{N}] label  -> {name}')
    download_file(lab_map[name]['id'], DEST/'labelsTr'/name)
    print(f'[{i}/{N}] DONE\n')

if split_id is not None:
    print('Downloading split.pkl...')
    download_file(split_id, DEST/'split.pkl')
    print('DONE split.pkl')
else:
    print('No split.pkl found in folder (that is OK, we will generate a tiny one).')

print('Tiny dataset ready at:', DEST)

[1/5] image  -> colon_001.nii.gz
SKIP: colon_001.nii.gz
[1/5] label  -> colon_001.nii.gz
SKIP: colon_001.nii.gz
[1/5] DONE

[2/5] image  -> colon_005.nii.gz
SKIP: colon_005.nii.gz
[2/5] label  -> colon_005.nii.gz
SKIP: colon_005.nii.gz
[2/5] DONE

[3/5] image  -> colon_006.nii.gz
SKIP: colon_006.nii.gz
[3/5] label  -> colon_006.nii.gz
SKIP: colon_006.nii.gz
[3/5] DONE

[4/5] image  -> colon_007.nii.gz
SKIP: colon_007.nii.gz
[4/5] label  -> colon_007.nii.gz
SKIP: colon_007.nii.gz
[4/5] DONE

[5/5] image  -> colon_008.nii.gz
SKIP: colon_008.nii.gz
[5/5] label  -> colon_008.nii.gz
SKIP: colon_008.nii.gz
[5/5] DONE

SKIP: split.pkl
DONE split.pkl
Tiny dataset ready at: /content/prism_data_tiny/colon


In [103]:
import pickle
from pathlib import Path

DATA_ROOT = Path('/content/prism_data_tiny/colon')
imgs = sorted([p.name for p in (DATA_ROOT/'imagesTr').glob('*.nii*')])

split = {
    'train': {n: [f'imagesTr/{n}', f'labelsTr/{n}'] for n in imgs},
    'val': {},
    'test': {},
}

with open(DATA_ROOT/'split_tiny.pkl', 'wb') as f:
    pickle.dump([split]*5, f)

# overwrite split.pkl to force dataset to use only tiny set
with open(DATA_ROOT/'split.pkl', 'wb') as f:
    pickle.dump([split]*5, f)

print('Wrote tiny split.pkl with', len(imgs), 'cases at:', DATA_ROOT/'split.pkl')
print('Cases:', imgs)

Wrote tiny split.pkl with 5 cases at: /content/prism_data_tiny/colon/split.pkl
Cases: ['colon_001.nii.gz', 'colon_005.nii.gz', 'colon_006.nii.gz', 'colon_007.nii.gz', 'colon_008.nii.gz']


In [104]:
import torch
import torchio as tio
from torch.utils.data import DataLoader
from prism3d.data.dataset import PrismCTDataset

DATA_ROOT = '/content/prism_data_tiny/colon'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, path = next(iter(dl))
print('path:', path[0])
print('img:', tuple(img.shape), img.dtype, 'min/max/mean:', float(img.min()), float(img.max()), float(img.mean()))
print('lab:', tuple(lab.shape), lab.dtype, 'unique:', torch.unique(lab))
print('lab sum:', float(lab.sum()))
print('len train:', len(ds))

path: /content/prism_data_tiny/colon/imagesTr/colon_008.nii.gz
img: (1, 1, 128, 128, 128) torch.float32 min/max/mean: -3.741824150085449 3.363581895828247 -1.3884979486465454
lab: (1, 1, 128, 128, 128) torch.float32 unique: metatensor([0., 1.])
lab sum: 13155.0
len train: 5


In [105]:
import os
import time
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.prism_sam3d import PrismSAM3D
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
    tio.RandomFlip(axes=(0, 1, 2)),
])

DATA_ROOT = '/content/prism_data_tiny/colon'
ds_train = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=train_tio)
dl_train = DataLoader(ds_train, batch_size=1, shuffle=True, num_workers=0)

model = PrismSAM3D().to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-5, weight_decay=0.01)

runner_train = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=3,           # start small
    num_clicks_train=10,   # start small
    dynamic_clicks=True,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,
)

epochs = 2
steps_per_epoch = 5  # since we only have 5 samples, this is basically one pass

for ep in range(1, epochs + 1):
    model.train()
    losses = []
    t0 = time.time()

    for step, (img, lab, _) in enumerate(dl_train):
        if step >= steps_per_epoch:
            break
        mean_loss, _ = runner_train.train_step(img, lab, opt=opt, scaler=scaler, autocast_ctx=autocast_ctx, grad_clip=1.0)
        losses.append(mean_loss)

    print(f'epoch {ep} | loss {sum(losses)/max(1,len(losses)):.4f} | time {time.time()-t0:.1f}s')

epoch 1 | loss 4.8359 | time 41.3s
epoch 2 | loss 4.2442 | time 41.7s


In [106]:
!pip -q install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib tqdm

In [107]:
import io
from pathlib import Path
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive = build('drive', 'v3')

WEIGHTS_FOLDER_ID = '1nPUC0cCsyA_w-tKkhL_Bw7lesBorGzCl'  # colon folder you gave

def list_children(parent_id: str):
    page_token = None
    out = []
    while True:
        res = drive.files().list(
            q=f'''('{parent_id}' in parents) and trashed=false''',
            fields='nextPageToken, files(id, name, mimeType, size)',
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        out.extend(res.get('files', []))
        page_token = res.get('nextPageToken')
        if not page_token:
            break
    return out

files = list_children(WEIGHTS_FOLDER_ID)
cands = [f for f in files if f['name'].endswith('.pth.tar') and 'best' in f['name'].lower()]

if len(cands) == 0:
    print('No best*.pth.tar found. Here are the files in the folder:')
    for f in sorted(files, key=lambda x: x['name']):
        print('-', f['name'])
    raise FileNotFoundError('Could not find best.pth.tar in the weights folder.')

# If there are multiple, pick the first alphabetically
cands = sorted(cands, key=lambda x: x['name'])
best_file = cands[0]

BEST_FILE_ID = best_file['id']
BEST_FILE_NAME = best_file['name']
BEST_FILE_SIZE = int(best_file.get('size', '0') or '0')

print('Found:', BEST_FILE_NAME)
print('File ID:', BEST_FILE_ID)
print('Size (bytes):', BEST_FILE_SIZE)

Found: best.pth.tar
File ID: 1R-cfIj0234-L0mqG7qT9bNTanWcq5RPs
Size (bytes): 1418347293


In [110]:
from pathlib import Path
from tqdm import tqdm
from googleapiclient.http import MediaIoBaseDownload

OUT_DIR = Path('/content/prism_pretrained/colon')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / BEST_FILE_NAME

def download_with_progress(file_id: str, out_path: Path, total_bytes: int | None = None):
    if out_path.exists() and out_path.stat().st_size > 0:
        print('Already exists:', out_path)
        return out_path

    request = drive.files().get_media(fileId=file_id)
    with io.FileIO(out_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 16)
        pbar = tqdm(total=total_bytes if total_bytes and total_bytes > 0 else None, unit='B', unit_scale=True, desc=out_path.name)
        last = 0
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status is not None:
                # status.progress() is 0..1
                if total_bytes and total_bytes > 0:
                    cur = int(status.progress() * total_bytes)
                    pbar.update(cur - last)
                    last = cur
        if total_bytes and total_bytes > 0:
            # ensure we end at total
            pbar.update(total_bytes - last)
        pbar.close()
    return out_path

out = download_with_progress(BEST_FILE_ID, OUT_PATH, BEST_FILE_SIZE)
print('Downloaded to:', out)
print('Local size:', out.stat().st_size)

Already exists: /content/prism_pretrained/colon/best.pth.tar
Downloaded to: /content/prism_pretrained/colon/best.pth.tar
Local size: 1418347293


In [112]:
from pathlib import Path

p = Path(OUT_PATH)
print('Path:', p)
print('Size bytes:', p.stat().st_size)

with open(p, 'rb') as f:
    head = f.read(256)

print('First 64 bytes:', head[:64])
if head.lstrip().startswith(b'<!DOCTYPE html') or head.lstrip().startswith(b'<html'):
    raise RuntimeError('This file looks like HTML (likely a Drive permission/virus-check page), not a real .pth.tar.')
else:
    print('Looks like a binary checkpoint file.')

Path: /content/prism_pretrained/colon/best.pth.tar
Size bytes: 1418347293
First 64 bytes: b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x12\x00archive/data.pklFB\x0e\x00ZZZZZZZZZZZZZZ'
Looks like a binary checkpoint file.


In [125]:
import torch

ckpt = torch.load(str(OUT_PATH), map_location='cpu', weights_only=False)
print('Loaded checkpoint keys:', list(ckpt.keys())[:30] if isinstance(ckpt, dict) else type(ckpt))

Loaded checkpoint keys: ['epoch', 'best_val_loss', 'model_state_dict', 'optimizer', 'lr_scheduler']


In [126]:
import sys
from pathlib import Path

REPO_DIR = Path('/content/PRISM_official')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/HaoLi12345/PRISM.git /content/PRISM_official

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Repo:', REPO_DIR)
print('Src :', SRC_DIR)
!ls -lah /content/PRISM_official/src/models | head

Repo: /content/PRISM_official
Src : /content/PRISM_official/src
total 92K
drwxr-xr-x 3 root root 4.0K Feb 24 12:05 .
drwxr-xr-x 9 root root 4.0K Feb 24 12:05 ..
-rw-r--r-- 1 root root 3.0K Feb 24 12:05 build_sam3D.py
-rw-r--r-- 1 root root  19K Feb 24 12:05 image_encoder.py
-rw-r--r-- 1 root root 7.4K Feb 24 12:05 mask_decoder.py
-rw-r--r-- 1 root root  13K Feb 24 12:05 prompt_encoder.py
drwxr-xr-x 2 root root 4.0K Feb 24 12:05 __pycache__
-rw-r--r-- 1 root root 7.2K Feb 24 12:05 sam3D.py
-rw-r--r-- 1 root root 9.0K Feb 24 12:05 transformer.py


In [127]:
import torch
from types import SimpleNamespace
from models.build_sam3D import sam_model_registry3D

args = SimpleNamespace(
    image_size=128,
    num_multiple_outputs=3,
    multiple_outputs=True,
    refine=True,              # <-- enable refine so keys match
    use_sam3d_turbo=False,
)

sam_refine = sam_model_registry3D['vit_b_ori'](args=args, checkpoint=None)

ckpt = torch.load(str(OUT_PATH), map_location='cpu', weights_only=False)
sd = extract_state_dict(ckpt)
sd2 = { (k[7:] if k.startswith('module.') else k): v for k, v in sd.items() }

missing, unexpected = sam_refine.load_state_dict(sd2, strict=False)
print('Missing:', len(missing), 'Unexpected:', len(unexpected))

sam_refine = sam_refine.cuda().eval()
model_refine = PRISMOfficialRunnerModel(sam_refine).cuda().eval()
print('Refine-enabled model ready.')

Unet_encoder features: (32, 32, 64, 128, 384, 32).
Unet_decoder features: (32, 32, 64, 128, 384, 32).
Missing: 0 Unexpected: 0
Refine-enabled model ready.


In [129]:
import sys
from pathlib import Path
from types import SimpleNamespace

REPO_DIR = Path('/content/PRISM_official')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/HaoLi12345/PRISM.git /content/PRISM_official

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models.build_sam3D import sam_model_registry3D

args = SimpleNamespace(
    image_size=128,
    num_multiple_outputs=3,
    multiple_outputs=True,
    refine=False,            # keep False for now; strict=False will ignore refine weights if present
    use_sam3d_turbo=False,
)

sam = sam_model_registry3D['vit_b_ori'](args=args, checkpoint=None)
print('sam built:', type(sam).__name__)

Unet_encoder features: (32, 32, 64, 128, 384, 32).
Unet_decoder features: (32, 32, 64, 128, 384, 32).
sam built: Sam3D


In [130]:
import torch

def extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for k in ['state_dict', 'model', 'net', 'network', 'model_state_dict', 'model_state']:
            if k in ckpt_obj and isinstance(ckpt_obj[k], dict):
                return ckpt_obj[k]
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj
    raise ValueError('Could not find a state_dict in this checkpoint.')

ckpt = torch.load(str(OUT_PATH), map_location='cpu', weights_only=False)
sd = extract_state_dict(ckpt)
sd2 = { (k[7:] if k.startswith('module.') else k): v for k, v in sd.items() }

missing, unexpected = sam.load_state_dict(sd2, strict=False)
print('Loaded into sam with strict=False')
print('Missing:', len(missing))
print('Unexpected:', len(unexpected))
print('Unexpected examples:', unexpected[:10])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = sam.to(device).eval()

Loaded into sam with strict=False
Missing: 0
Unexpected: 22
Unexpected examples: ['mask_decoder.refine.first_conv.weight', 'mask_decoder.refine.first_conv.bias', 'mask_decoder.refine.conv1.conv_0.conv.weight', 'mask_decoder.refine.conv1.conv_0.conv.bias', 'mask_decoder.refine.conv1.conv_0.adn.N.weight', 'mask_decoder.refine.conv1.conv_0.adn.N.bias', 'mask_decoder.refine.conv1.conv_1.conv.weight', 'mask_decoder.refine.conv1.conv_1.conv.bias', 'mask_decoder.refine.conv1.conv_1.adn.N.weight', 'mask_decoder.refine.conv1.conv_1.adn.N.bias']


In [131]:
import numpy as np
import torch

def extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for k in ['state_dict', 'model', 'net', 'network', 'model_state_dict', 'model_state']:
            if k in ckpt_obj and isinstance(ckpt_obj[k], dict):
                return ckpt_obj[k]
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj
    raise ValueError('Could not find a state_dict in this checkpoint.')

# If you trust the authors' weights (you do), this avoids the new PyTorch weights-only restriction
ckpt = torch.load(str(OUT_PATH), map_location='cpu', weights_only=False)
sd = extract_state_dict(ckpt)

# Strip DDP prefix if present
sd2 = {}
for k, v in sd.items():
    sd2[k[7:]] = v if k.startswith('module.') else v
    if not k.startswith('module.'):
        sd2[k] = v

# The above line duplicates keys; cleanly rebuild:
sd2 = { (k[7:] if k.startswith('module.') else k): v for k, v in sd.items() }

missing, unexpected = sam.load_state_dict(sd2, strict=False)
print('Loaded into official model with strict=False')
print('Missing:', len(missing))
print('Unexpected:', len(unexpected))
print('Missing examples:', missing[:25])
print('Unexpected examples:', unexpected[:25])

sam = sam.cuda().eval()

Loaded into official model with strict=False
Missing: 0
Unexpected: 22
Missing examples: []
Unexpected examples: ['mask_decoder.refine.first_conv.weight', 'mask_decoder.refine.first_conv.bias', 'mask_decoder.refine.conv1.conv_0.conv.weight', 'mask_decoder.refine.conv1.conv_0.conv.bias', 'mask_decoder.refine.conv1.conv_0.adn.N.weight', 'mask_decoder.refine.conv1.conv_0.adn.N.bias', 'mask_decoder.refine.conv1.conv_1.conv.weight', 'mask_decoder.refine.conv1.conv_1.conv.bias', 'mask_decoder.refine.conv1.conv_1.adn.N.weight', 'mask_decoder.refine.conv1.conv_1.adn.N.bias', 'mask_decoder.refine.conv2.conv_0.conv.weight', 'mask_decoder.refine.conv2.conv_0.conv.bias', 'mask_decoder.refine.conv2.conv_0.adn.N.weight', 'mask_decoder.refine.conv2.conv_0.adn.N.bias', 'mask_decoder.refine.conv2.conv_1.conv.weight', 'mask_decoder.refine.conv2.conv_1.conv.bias', 'mask_decoder.refine.conv2.conv_1.adn.N.weight', 'mask_decoder.refine.conv2.conv_1.adn.N.bias', 'mask_decoder.refine.conv_error_map.weight', '

In [132]:
import torch
import torch.nn as nn

class ImageEncoderForRunner(nn.Module):
    def __init__(self, official_image_encoder: nn.Module):
        super().__init__()
        self.enc = official_image_encoder

    def forward(self, x: torch.Tensor):
        # official returns: (image_embedding, feature_list)
        image_emb, feature_list = self.enc(x)

        # Our runner expects: (image_emb, up_emb, feature_list)
        # In official code, "feature_list" plays the role of upscaled embedding inputs to decoder.
        up_emb = feature_list
        return image_emb, up_emb, feature_list


class PRISMOfficialRunnerModel(nn.Module):
    def __init__(self, official_sam: nn.Module):
        super().__init__()
        self.image_encoder = ImageEncoderForRunner(official_sam.image_encoder)
        self.prompt_encoder = official_sam.prompt_encoder
        self.mask_decoder = official_sam.mask_decoder

    def step(
        self,
        image_embedding: torch.Tensor,
        upscaled_embedding,
        prev_mask_prob: torch.Tensor,
        points=None,
        boxes=None,
        multimask_output: bool = True,
    ):
        # official prompt encoder takes image_embeddings and returns (prompt_emb, new_img_emb)
        prompt_emb, new_img_emb = self.prompt_encoder(
            points=points,
            boxes=boxes,
            masks=prev_mask_prob,
            image_embeddings=image_embedding,
        )

        # official mask decoder takes (prompt_embeddings, image_embeddings, feature_list)
        masks, scores = self.mask_decoder(
            prompt_embeddings=prompt_emb,
            image_embeddings=new_img_emb,
            feature_list=upscaled_embedding,
        )

        # The official decoder already outputs multiple masks + scores
        return masks, scores

print('Wrapper classes defined.')

Wrapper classes defined.


In [134]:
import torch
import torch.nn as nn

def infer_mask_prompt_input_size(sam_model, image_size=128):
    # get downscaler module (name differs sometimes)
    pe = sam_model.prompt_encoder
    candidates = []
    for name in ['mask_downscaler', 'mask_downscaling', 'mask_downscaling_layer', 'mask_downscale']:
        if hasattr(pe, name):
            candidates.append(getattr(pe, name))
    if len(candidates) == 0:
        # brute search: first nn.Module that contains Conv3d and looks like a downscaler
        for m in pe.modules():
            if isinstance(m, nn.Conv3d):
                # parent module is hard to get; fallback later
                break
        raise RuntimeError('Could not find mask downscaler module on official prompt_encoder.')

    down = candidates[0]

    # multiply stride of Conv3d layers
    stride_total = 1
    for m in down.modules():
        if isinstance(m, nn.Conv3d):
            s = m.stride
            if isinstance(s, tuple):
                s = max(int(s[0]), int(s[1]), int(s[2]))
            stride_total *= int(s)

    # the downscaler is designed so that: mask_input / stride_total = image_embedding_grid
    # official ViT grid is usually 8 for image_size=128 -> patch size 16
    grid = 8
    expected_mask_in = grid * stride_total
    return stride_total, expected_mask_in

stride_total, expected_mask_in = infer_mask_prompt_input_size(sam, image_size=128)
print('mask downscaler stride_total:', stride_total)
print('expected mask prompt input size:', expected_mask_in)

mask downscaler stride_total: 4
expected mask prompt input size: 32


In [135]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImageEncoderForRunner(nn.Module):
    def __init__(self, official_image_encoder: nn.Module):
        super().__init__()
        self.enc = official_image_encoder

    def forward(self, x: torch.Tensor):
        # official returns: (image_embedding, feature_list)
        image_emb, feature_list = self.enc(x)
        # our runner expects (image_emb, upscaled_embedding, feature_list)
        return image_emb, feature_list, feature_list


class PRISMOfficialRunnerModel(nn.Module):
    def __init__(self, official_sam: nn.Module):
        super().__init__()
        self.image_encoder = ImageEncoderForRunner(official_sam.image_encoder)
        self.prompt_encoder = official_sam.prompt_encoder
        self.mask_decoder = official_sam.mask_decoder

        # detect expected mask prompt input size
        self.mask_stride_total, self.mask_prompt_in_size = infer_mask_prompt_input_size(official_sam, image_size=128)

    def _prep_boxes(self, boxes):
        if boxes is None:
            return None
        # runner gives (B,6) => convert to (B,2,3) if needed
        if boxes.dim() == 2 and boxes.shape[1] == 6:
            b0 = boxes[:, 0:3]
            b1 = boxes[:, 3:6]
            return torch.stack([b0, b1], dim=1)
        return boxes

    def _prep_mask_prompt(self, prev_mask_prob, image_embedding):
        if prev_mask_prob is None:
            return None
        # ensure float32 for stable downsampling
        m = prev_mask_prob.float()

        # downsample to expected input size for official prompt encoder mask branch
        target = int(self.mask_prompt_in_size)

        # only resize if needed
        if m.shape[-1] != target or m.shape[-2] != target or m.shape[-3] != target:
            m = F.interpolate(m, size=(target, target, target), mode='trilinear', align_corners=False)
        return m

    def step(
        self,
        image_embedding: torch.Tensor,
        upscaled_embedding,
        prev_mask_prob: torch.Tensor,
        points=None,
        boxes=None,
        multimask_output: bool = True,
    ):
        boxes = self._prep_boxes(boxes)
        masks_in = self._prep_mask_prompt(prev_mask_prob, image_embedding)

        prompt_emb, new_img_emb = self.prompt_encoder(
            points=points,
            boxes=boxes,
            masks=masks_in,
            image_embeddings=image_embedding,
        )

        masks, scores = self.mask_decoder(
            prompt_embeddings=prompt_emb,
            image_embeddings=new_img_emb,
            feature_list=upscaled_embedding,
        )

        return masks, scores

print('Fixed wrapper ready.')

Fixed wrapper ready.


In [136]:
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.engine.interactive_runner import InteractiveRunner3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = PRISMOfficialRunnerModel(sam).to(device).eval()

DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1),
])

ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)

runner_val = InteractiveRunner3D(
    model=model,
    device=device,
    iter_nums=11,
    num_clicks_val=10,
    dynamic_clicks=False,
    use_box=True,
    dynamic_box=False,
    boundary_kernel_size=5,
    multiple_outputs=True,
    detach_prev_mask=True,
    freeze_image_encoder=True,
)

dice_list = []
curve_sum = None
n = 0

with torch.no_grad():
    for img, lab, _ in dl:
        mean_dice, _, curve = runner_val.run(img, lab, train=False, return_curve=True)
        dice_list.append(float(mean_dice.detach().cpu()))
        cvals = [float(x.detach().cpu()) for x in curve]
        curve_sum = cvals if curve_sum is None else [a + b for a, b in zip(curve_sum, cvals)]
        n += 1

mean_dice = sum(dice_list) / max(1, len(dice_list))
mean_curve = [c / max(1, n) for c in curve_sum]

print('Mean dice (5 cases):', mean_dice)
print('Mean dice curve:')
for i, d in enumerate(mean_curve, start=1):
    print(f'  iter {i:02d}: {d:.4f}')

Mean dice (5 cases): 0.8309815406799317
Mean dice curve:
  iter 01: 0.8359
  iter 02: 0.8382
  iter 03: 0.8320
  iter 04: 0.8295
  iter 05: 0.8237
  iter 06: 0.8310
  iter 07: 0.8328
  iter 08: 0.8282
  iter 09: 0.8301
  iter 10: 0.8300
  iter 11: 0.8293


In [137]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'voxel_attention_head.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalBCEWithLogits(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1.0 - p) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        loss = alpha_t * ((1.0 - p_t) ** self.gamma) * bce
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


class VoxelAttentionHead3D(nn.Module):
    def __init__(self, feat_ch: int = 32, mid_ch: int = 32):
        super().__init__()
        in_ch = int(feat_ch) + 1  # + predicted mask prob
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, mid_ch, kernel_size=3, padding=1),
            nn.GroupNorm(8, mid_ch),
            nn.GELU(),
            nn.Conv3d(mid_ch, mid_ch, kernel_size=3, padding=1),
            nn.GroupNorm(8, mid_ch),
            nn.GELU(),
            nn.Conv3d(mid_ch, 1, kernel_size=1),
        )

    def forward(self, feat_128: torch.Tensor, mask_prob: torch.Tensor) -> torch.Tensor:
        # feat_128: (B,C,128,128,128)
        # mask_prob: (B,1,128,128,128)
        x = torch.cat([feat_128, mask_prob], dim=1)
        return self.net(x)
''')

print('Wrote:', PKG_ROOT/'models'/'voxel_attention_head.py')

Wrote: /content/prism3d_project/prism3d/models/voxel_attention_head.py


In [138]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'attention_utils.py', '''
import torch
import torch.nn.functional as F


def binary_dilate_3d(x: torch.Tensor, w: int) -> torch.Tensor:
    # x: (B,1,D,H,W) bool/0-1
    k = 2 * int(w) + 1
    x = x.float()
    y = F.max_pool3d(x, kernel_size=k, stride=1, padding=int(w))
    return y > 0.5


def binary_erode_3d(x: torch.Tensor, w: int) -> torch.Tensor:
    # erosion via complement + dilation
    k = 2 * int(w) + 1
    x = x.float()
    y = 1.0 - F.max_pool3d(1.0 - x, kernel_size=k, stride=1, padding=int(w))
    return y > 0.5


def build_mixed_attention_target(mask_logits: torch.Tensor, y: torch.Tensor, w: int = 3, alpha: float = 0.7) -> torch.Tensor:
    with torch.no_grad():
        prob = torch.sigmoid(mask_logits)
        b_hat = (prob > 0.5)
        yb = (y > 0.5)

        e_full = (b_hat ^ yb)  # bool

        dil = binary_dilate_3d(yb, w=w)
        ero = binary_erode_3d(yb, w=w)
        band = dil & (~ero)  # boundary band

        e_bd = e_full & band

        # Mixed target: boundary errors -> 1.0, interior errors -> (1-alpha), correct -> 0
        T = e_full.float() * (1.0 - float(alpha) + float(alpha) * band.float())
        # safety clamp
        T = T.clamp(0.0, 1.0)

    return T


def greedy_min_dist_pick(coords_zyx: torch.Tensor, scores: torch.Tensor, n_pick: int, min_dist: int) -> torch.Tensor:
    order = torch.argsort(scores, descending=True)
    picked = []
    for idx in order.tolist():
        c = coords_zyx[idx]
        ok = True
        for j in picked:
            cj = coords_zyx[j]
            if torch.abs(c - cj).max().item() < int(min_dist):
                ok = False
                break
        if ok:
            picked.append(idx)
        if len(picked) >= int(n_pick):
            break
    return torch.tensor(picked, dtype=torch.long)


def sample_clicks_from_attention(
    attn_prob: torch.Tensor,
    mask_logits: torch.Tensor,
    y: torch.Tensor,
    num_clicks: int,
    topk: int = 2048,
    min_dist: int = 12,
    eps: float = 1e-8,
):
    device = y.device
    B = y.shape[0]

    with torch.no_grad():
        prob = torch.sigmoid(mask_logits)
        pred = (prob > 0.5)
        gt = (y > 0.5)
        e_full = (pred ^ gt)  # bool

    # CPU sampling to avoid GPU temp spikes
    attn_cpu = attn_prob[:, 0].detach().to('cpu')
    e_cpu = e_full[:, 0].detach().to('cpu')
    gt_cpu = gt[:, 0].detach().to('cpu')
    pred_cpu = pred[:, 0].detach().to('cpu')

    batch_coords = []
    batch_labs = []

    for b in range(B):
        if torch.count_nonzero(e_cpu[b]).item() == 0:
            # no error -> return empty, caller should fallback
            batch_coords.append(torch.zeros((1,0,3), dtype=torch.float32))
            batch_labs.append(torch.zeros((1,0), dtype=torch.long))
            continue

        score_map = attn_cpu[b] * e_cpu[b].float()
        flat = score_map.flatten()

        K = min(int(topk), flat.numel())
        vals, inds = torch.topk(flat, k=K, largest=True, sorted=True)

        # filter tiny scores
        keep = vals > (vals.max().item() * 0.05 + eps)
        vals = vals[keep]
        inds = inds[keep]
        if vals.numel() == 0:
            batch_coords.append(torch.zeros((1,0,3), dtype=torch.float32))
            batch_labs.append(torch.zeros((1,0), dtype=torch.long))
            continue

        D, H, W = score_map.shape
        z = inds // (H * W)
        y_ = (inds % (H * W)) // W
        x = inds % W
        coords = torch.stack([z, y_, x], dim=1).to(torch.int64)  # (K,3)

        picked = greedy_min_dist_pick(coords, vals, n_pick=num_clicks, min_dist=min_dist)
        coords_sel = coords[picked]  # (N,3)

        labs_sel = []
        for c in coords_sel:
            zz, yy, xx = int(c[0]), int(c[1]), int(c[2])
            # label by error sign: FN->pos, FP->neg, else fallback to gt
            if gt_cpu[b, zz, yy, xx].item() == 1 and pred_cpu[b, zz, yy, xx].item() == 0:
                labs_sel.append(1)
            elif gt_cpu[b, zz, yy, xx].item() == 0 and pred_cpu[b, zz, yy, xx].item() == 1:
                labs_sel.append(0)
            else:
                labs_sel.append(1 if gt_cpu[b, zz, yy, xx].item() == 1 else 0)

        coords_out = coords_sel.float().unsqueeze(0)  # (1,N,3)
        labs_out = torch.tensor(labs_sel, dtype=torch.long).unsqueeze(0)  # (1,N)

        batch_coords.append(coords_out)
        batch_labs.append(labs_out)

    coords = torch.cat(batch_coords, dim=0).to(device=device)
    labs = torch.cat(batch_labs, dim=0).to(device=device)
    return coords, labs
''')

print('Wrote:', PKG_ROOT/'engine'/'attention_utils.py')

Wrote: /content/prism3d_project/prism3d/engine/attention_utils.py


In [139]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'engine'/'interactive_runner_attn.py', '''
import torch
from prism3d.engine.interactive_utils import sample_click_points  # fallback sampler
from prism3d.engine.attention_utils import build_mixed_attention_target, sample_clicks_from_attention
from prism3d.models.voxel_attention_head import VoxelAttentionHead3D, FocalBCEWithLogits


class InteractiveRunner3D_Attn:
    def __init__(
        self,
        model,                       # PRISM runner-compatible model (frozen)
        attention_head: VoxelAttentionHead3D,
        device: str = 'cuda',
        iter_nums: int = 11,
        num_clicks_train: int = 50,
        num_clicks_val: int = 10,
        alpha_mix: float = 0.7,
        band_w: int = 3,
        lambda_att: float = 1.0,
        topk: int = 2048,
        min_dist: int = 12,
        use_box: bool = True,
        dynamic_box: bool = False,
    ):
        self.model = model
        self.attn = attention_head
        self.device = device

        self.iter_nums = int(iter_nums)
        self.num_clicks_train = int(num_clicks_train)
        self.num_clicks_val = int(num_clicks_val)

        self.alpha_mix = float(alpha_mix)
        self.band_w = int(band_w)
        self.lambda_att = float(lambda_att)

        self.topk = int(topk)
        self.min_dist = int(min_dist)

        self.use_box = use_box
        self.dynamic_box = dynamic_box

        self.focal = FocalBCEWithLogits(alpha=0.25, gamma=2.0)

        # freeze PRISM (Phase 1)
        for p in self.model.parameters():
            p.requires_grad_(False)

        for p in self.attn.parameters():
            p.requires_grad_(True)

    def _pick_feat128(self, upscaled_embedding):
        # upscaled_embedding may be list/tuple of tensors or a tensor
        if torch.is_tensor(upscaled_embedding):
            return upscaled_embedding
        # find tensor with spatial 128^3
        for t in upscaled_embedding:
            if torch.is_tensor(t) and t.dim() == 5 and t.shape[-1] == 128 and t.shape[-2] == 128 and t.shape[-3] == 128:
                return t
        # fallback: last tensor
        for t in reversed(upscaled_embedding):
            if torch.is_tensor(t) and t.dim() == 5:
                return t
        raise RuntimeError('Could not find a 3D feature tensor in upscaled_embedding.')

    def train_step(
        self,
        image: torch.Tensor,
        label: torch.Tensor,
        opt,
        scaler,
        autocast_ctx,
        grad_clip: float = 1.0,
        debug: bool = True,
    ):
        image = image.to(self.device)
        label = label.to(self.device)

        opt.zero_grad(set_to_none=True)

        # image features (frozen)
        with torch.no_grad():
            image_emb, up_emb, _ = self.model.image_encoder(image)

        feat128 = self._pick_feat128(up_emb).detach()  # (B,C,128,128,128)

        prev_logits = torch.zeros_like(label, dtype=torch.float32, device=self.device)

        loss_att_sum = 0.0

        # prompt history (points + labels)
        points_hist = None
        labels_hist = None

        for it in range(self.iter_nums):
            prev_prob = torch.sigmoid(prev_logits) if it > 0 else prev_logits
            prev_prob = prev_prob.detach()

            # build box if the underlying model uses it via runner (optional)
            # we keep box logic in the model.step via passed boxes (your wrapper already converts)
            boxes = None
            if self.use_box:
                # (B,6) z0,y0,x0,z1,y1,x1 from GT
                gt = (label[:, 0] > 0)
                # simple bbox from gt without randomness (you can swap in your bbox function if needed)
                bboxes = []
                for vol in gt:
                    i_any = vol.any(dim=2).any(dim=1)
                    j_any = vol.any(dim=2).any(dim=0)
                    k_any = vol.any(dim=1).any(dim=0)
                    i_min, i_max = torch.where(i_any)[0][[0, -1]]
                    j_min, j_max = torch.where(j_any)[0][[0, -1]]
                    k_min, k_max = torch.where(k_any)[0][[0, -1]]
                    bb = torch.tensor([int(i_min), int(j_min), int(k_min), int(i_max)+1, int(j_max)+1, int(k_max)+1], device=label.device)
                    bboxes.append(bb)
                boxes = torch.stack(bboxes, dim=0).float()

            # PRISM forward (frozen)
            with torch.no_grad():
                masks, scores = self.model.step(
                    image_embedding=image_emb,
                    upscaled_embedding=up_emb,
                    prev_mask_prob=prev_prob,
                    points=None if points_hist is None else (points_hist, labels_hist),
                    boxes=boxes,
                    multimask_output=True,
                )
                best_idx = torch.argmax(scores, dim=1)
                best = torch.stack([masks[b, best_idx[b]] for b in range(masks.shape[0])], dim=0).unsqueeze(1)

            # attention target
            T = build_mixed_attention_target(best, label, w=self.band_w, alpha=self.alpha_mix)  # (B,1,128,128,128)

            # attention prediction (trainable head)
            with autocast_ctx():
                attn_logits = self.attn(feat128, torch.sigmoid(best).detach())
                loss_att = self.focal(attn_logits, T) * self.lambda_att / float(self.iter_nums)

            if scaler is not None:
                scaler.scale(loss_att).backward()
            else:
                loss_att.backward()

            loss_att_sum += float(loss_att.detach().cpu())

            # guided sampling (non-diff)
            attn_prob = torch.sigmoid(attn_logits).detach()

            coords, labs = sample_clicks_from_attention(
                attn_prob=attn_prob,
                mask_logits=best.detach(),
                y=label,
                num_clicks=self.num_clicks_train,
                topk=self.topk,
                min_dist=self.min_dist,
            )

            # fallback if attention sampler returned empty clicks
            if coords.shape[1] == 0:
                coords, labs = sample_click_points(
                    prev_prob=torch.sigmoid(best).detach(),
                    label=label,
                    num_clicks=self.num_clicks_train,
                    dynamic=True,
                    mode='train',
                )

            # accumulate points (append)
            if points_hist is None:
                points_hist = coords
                labels_hist = labs
            else:
                points_hist = torch.cat([points_hist, coords], dim=1)
                labels_hist = torch.cat([labels_hist, labs], dim=1)

            prev_logits = best.detach()

            if debug and it == 0:
                with torch.no_grad():
                    a = attn_prob
                    print('debug attn mean/max:', float(a.mean()), float(a.max()))
                    print('debug T mean/max   :', float(T.mean()), float(T.max()))
                    if T.sum() > 0:
                        print('debug attn on T>0  :', float(a[T>0].mean()))
                    if coords.shape[1] > 0:
                        z, y_, x = [int(v) for v in coords[0, 0].tolist()]
                        print('debug sampled zyx  :', (z, y_, x), 'attn@pt:', float(a[0,0,z,y_,x]))

        if scaler is not None:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(self.attn.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(self.attn.parameters(), grad_clip)
            opt.step()

        return loss_att_sum
''')

print('Wrote:', PKG_ROOT/'engine'/'interactive_runner_attn.py')

Wrote: /content/prism3d_project/prism3d/engine/interactive_runner_attn.py


In [140]:
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset
from prism3d.models.voxel_attention_head import VoxelAttentionHead3D
from prism3d.engine.interactive_runner_attn import InteractiveRunner3D_Attn

device = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_ROOT = '/content/prism_data_tiny/colon'
val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))

# Your pretrained PRISM model wrapper is `model` (PRISMOfficialRunnerModel) from earlier.
# Ensure it is on device and eval/frozen.
model = model.to(device).eval()

# Build attention head
attn_head = VoxelAttentionHead3D(feat_ch=32, mid_ch=32).to(device).train()
opt_attn = torch.optim.AdamW(attn_head.parameters(), lr=1e-4, weight_decay=1e-2)

runner_attn = InteractiveRunner3D_Attn(
    model=model,
    attention_head=attn_head,
    device=device,
    iter_nums=3,            # sanity first
    num_clicks_train=10,    # sanity first
    alpha_mix=0.7,
    band_w=3,
    lambda_att=1.0,
    topk=1024,
    min_dist=12,
    use_box=True,
)

loss_att = runner_attn.train_step(
    image=img,
    label=lab,
    opt=opt_attn,
    scaler=scaler,
    autocast_ctx=autocast_ctx,
    grad_clip=1.0,
    debug=True,
)

print('Attention training step done. loss_att_sum:', loss_att)

debug attn mean/max: 0.568359375 0.8798828125
debug T mean/max   : 0.0008555889362469316 1.0
debug attn on T>0  : 0.52978515625
debug sampled zyx  : (40, 56, 63) attn@pt: 0.75390625
Attention training step done. loss_att_sum: 0.2121603786945343


In [141]:
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

attn_head.train()
model.eval()

losses = []
for step, (img, lab, _) in enumerate(dl):
    l = runner_attn.train_step(img, lab, opt=opt_attn, scaler=scaler, autocast_ctx=autocast_ctx, grad_clip=1.0, debug=(step==0))
    losses.append(l)

print('Epoch done. mean loss:', sum(losses)/max(1,len(losses)))

debug attn mean/max: 0.5361328125 0.90576171875
debug T mean/max   : 0.002929210662841797 1.0
debug attn on T>0  : 0.521484375
debug sampled zyx  : (43, 80, 85) attn@pt: 0.7841796875
Epoch done. mean loss: 0.11540982015430927


In [142]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path('/content/prism3d_project')
PKG_ROOT = PROJECT_ROOT / 'prism3d'

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'voxel_attention_head.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalBCEWithLogits(nn.Module):
    def __init__(self, alpha_pos: float = 0.75, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        p = torch.sigmoid(logits)
        p_t = p * targets + (1.0 - p) * (1.0 - targets)

        # weight positives higher (pos are very sparse)
        alpha_t = self.alpha_pos * targets + (1.0 - self.alpha_pos) * (1.0 - targets)

        loss = alpha_t * ((1.0 - p_t) ** self.gamma) * bce

        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


class VoxelAttentionHead3D(nn.Module):
    def __init__(self, feat_ch: int = 32, mid_ch: int = 32, init_bias: float = -4.0):
        super().__init__()
        in_ch = int(feat_ch) + 1  # + predicted mask prob

        self.conv1 = nn.Conv3d(in_ch, mid_ch, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(8, mid_ch)
        self.conv2 = nn.Conv3d(mid_ch, mid_ch, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(8, mid_ch)
        self.out = nn.Conv3d(mid_ch, 1, kernel_size=1)

        self.act = nn.GELU()

        # init: sparse attention at start
        nn.init.zeros_(self.out.weight)
        nn.init.constant_(self.out.bias, float(init_bias))

    def forward(self, feat_128: torch.Tensor, mask_prob: torch.Tensor) -> torch.Tensor:
        x = torch.cat([feat_128, mask_prob], dim=1)
        x = self.act(self.gn1(self.conv1(x)))
        x = self.act(self.gn2(self.conv2(x)))
        return self.out(x)
''')

print('Patched:', PKG_ROOT/'models'/'voxel_attention_head.py')

Patched: /content/prism3d_project/prism3d/models/voxel_attention_head.py


In [145]:
from pathlib import Path
import importlib

p = Path('/content/prism3d_project/prism3d/engine/interactive_runner_attn.py')
txt = p.read_text(encoding='utf-8')

old = 'self.focal = FocalBCEWithLogits(alpha=0.25, gamma=2.0)'
new = 'self.focal = FocalBCEWithLogits(alpha_pos=0.75, gamma=2.0)'

if old not in txt:
    # fallback: try to replace any FocalBCEWithLogits(...) line
    import re
    txt2 = re.sub(r'''self\.focal\s*=\s*FocalBCEWithLogits\([^\)]*\)''', new, txt)
    if txt2 == txt:
        raise RuntimeError('Could not find focal loss init line to patch.')
    txt = txt2
else:
    txt = txt.replace(old, new)

p.write_text(txt, encoding='utf-8')
print('Patched:', p)

import prism3d.engine.interactive_runner_attn as ira
importlib.reload(ira)

from prism3d.engine.interactive_runner_attn import InteractiveRunner3D_Attn
print('Reloaded InteractiveRunner3D_Attn OK')

Patched: /content/prism3d_project/prism3d/engine/interactive_runner_attn.py
Reloaded InteractiveRunner3D_Attn OK


In [146]:
import torch
import torchio as tio
from torch.utils.data import DataLoader

from prism3d.data.dataset import PrismCTDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))

# PRISM pretrained model wrapper should be in `model`
model = model.to(device).eval()

attn_head = VoxelAttentionHead3D(feat_ch=32, mid_ch=32, init_bias=-4.0).to(device).train()
opt_attn = torch.optim.AdamW(attn_head.parameters(), lr=2e-4, weight_decay=1e-2)

runner_attn = InteractiveRunner3D_Attn(
    model=model,
    attention_head=attn_head,
    device=device,
    iter_nums=3,
    num_clicks_train=1,      # next-click recommendation
    alpha_mix=0.7,
    band_w=3,
    lambda_att=1.0,
    topk=1024,
    min_dist=12,
    use_box=True,
)

loss_att = runner_attn.train_step(
    image=img,
    label=lab,
    opt=opt_attn,
    scaler=scaler,
    autocast_ctx=autocast_ctx,
    grad_clip=1.0,
    debug=True,
)

print('loss_att_sum:', loss_att)

debug attn mean/max: 0.0179901123046875 0.0179901123046875
debug T mean/max   : 0.0008174419635906816 1.0
debug attn on T>0  : 0.0179901123046875
debug sampled zyx  : (51, 68, 73) attn@pt: 0.0179901123046875
loss_att_sum: 0.0026068674051202834


In [147]:
import torch
import torchio as tio
from torch.utils.data import DataLoader
from prism3d.data.dataset import PrismCTDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

model.eval()
attn_head.train()

losses = []
for step, (img, lab, _) in enumerate(dl):
    l = runner_attn.train_step(img, lab, opt=opt_attn, scaler=scaler, autocast_ctx=autocast_ctx, grad_clip=1.0, debug=(step==0))
    losses.append(l)
    if step >= 9:
        break

print('10-step mean loss:', sum(losses)/len(losses))

debug attn mean/max: 0.0180206298828125 0.018157958984375
debug T mean/max   : 0.004642915911972523 1.0
debug attn on T>0  : 0.01800537109375
debug sampled zyx  : (62, 38, 93) attn@pt: 0.018096923828125
10-step mean loss: 0.00901798652485013


In [155]:
from pathlib import Path
import textwrap
import importlib

PKG_ROOT = Path('/content/prism3d_project/prism3d')

def write_py(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

write_py(PKG_ROOT/'models'/'voxel_attention_head.py', '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalBCEWithLogits(nn.Module):
    def __init__(self, alpha_pos: float = 0.75, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        p = torch.sigmoid(logits)
        p_t = p * targets + (1.0 - p) * (1.0 - targets)

        alpha_t = self.alpha_pos * targets + (1.0 - self.alpha_pos) * (1.0 - targets)
        loss = alpha_t * ((1.0 - p_t) ** self.gamma) * bce

        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


class VoxelAttentionHead3D(nn.Module):
    def __init__(self, feat_ch: int = 32, mid_ch: int = 32, init_bias: float = -4.0, out_w_std: float = 1e-3):
        super().__init__()
        in_ch = int(feat_ch) + 1

        self.conv1 = nn.Conv3d(in_ch, mid_ch, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(8, mid_ch)
        self.conv2 = nn.Conv3d(mid_ch, mid_ch, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(8, mid_ch)
        self.out = nn.Conv3d(mid_ch, 1, kernel_size=1)

        self.act = nn.GELU()

        # critical: out weights must NOT be zero, or gradients won't reach conv1/conv2
        nn.init.normal_(self.out.weight, mean=0.0, std=float(out_w_std))
        nn.init.constant_(self.out.bias, float(init_bias))

    def forward(self, feat_128: torch.Tensor, mask_prob: torch.Tensor) -> torch.Tensor:
        x = torch.cat([feat_128, mask_prob], dim=1)
        x = self.act(self.gn1(self.conv1(x)))
        x = self.act(self.gn2(self.conv2(x)))
        return self.out(x)
''')

import prism3d.models.voxel_attention_head as vah
importlib.reload(vah)
from prism3d.models.voxel_attention_head import VoxelAttentionHead3D, FocalBCEWithLogits

print('Patched + reloaded voxel_attention_head.py')

Patched + reloaded voxel_attention_head.py


In [152]:
import torch
import torchio as tio
from torch.utils.data import DataLoader
from prism3d.data.dataset import PrismCTDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))

model = model.to(device).eval()

attn_head = VoxelAttentionHead3D(feat_ch=32, mid_ch=32, init_bias=-4.0, out_w_std=1e-3).to(device).train()
opt_attn = torch.optim.AdamW(attn_head.parameters(), lr=5e-4, weight_decay=1e-2)

runner_attn = InteractiveRunner3D_Attn(
    model=model,
    attention_head=attn_head,
    device=device,
    iter_nums=3,
    num_clicks_train=1,
    alpha_mix=0.7,
    band_w=3,
    lambda_att=5.0,
    topk=1024,
    min_dist=12,
    use_box=True,
)

loss_att = runner_attn.train_step(
    image=img,
    label=lab,
    opt=opt_attn,
    scaler=scaler,
    autocast_ctx=autocast_ctx,
    grad_clip=1.0,
    debug=True,
)

print('loss_att_sum:', loss_att)

debug attn mean/max: 0.017961520701646805 0.018305690959095955
debug T mean/max   : 0.004036045167595148 1.0
debug attn on T>0  : 0.017985941842198372
debug sampled zyx  : (67, 66, 62) attn@pt: 0.018208125606179237
loss_att_sum: 0.055329496040940285


In [157]:
from pathlib import Path
import re
import importlib

p = Path('/content/prism3d_project/prism3d/engine/interactive_runner_attn.py')
txt = p.read_text(encoding='utf-8')

# 1) ensure F is imported
if 'import torch.nn.functional as F' not in txt:
    txt = txt.replace('import torch\n', 'import torch\nimport torch.nn.functional as F\n', 1)

# 2) add rank hyperparams in __init__
if 'self.rank_weight' not in txt:
    txt = txt.replace(
        'self.lambda_att = float(lambda_att)\n',
        '''self.lambda_att = float(lambda_att)
        self.rank_weight = 0.2   # ranking term strength
        self.rank_samples = 2048 # voxels per batch item for ranking
''',
        1
    )

# 3) replace the fp32 attention block to include ranking loss
pattern = r'''with torch\.cuda\.amp\.autocast\(enabled=False\):\s*\n\s*attn_logits = self\.attn\(feat128\.float\(\), torch\.sigmoid\(best\)\.detach\(\)\.float\(\)\)\s*\n\s*loss_att = self\.focal\(attn_logits, T\.float\(\)\) \* self\.lambda_att / float\(self\.iter_nums\)'''

replacement = '''with torch.cuda.amp.autocast(enabled=False):
                attn_logits = self.attn(feat128.float(), torch.sigmoid(best).detach().float())

                # focal term
                loss_focal = self.focal(attn_logits, T.float())

                # ranking term: encourage attn(pos) > attn(neg)
                loss_rank = torch.tensor(0.0, device=attn_logits.device, dtype=attn_logits.dtype)
                B = attn_logits.shape[0]
                Aflat = attn_logits.view(B, -1)
                Tflat = T.view(B, -1)

                for b in range(B):
                    pos = torch.nonzero(Tflat[b] > 0, as_tuple=False).squeeze(1)
                    neg = torch.nonzero(Tflat[b] == 0, as_tuple=False).squeeze(1)
                    if pos.numel() == 0 or neg.numel() == 0:
                        continue
                    n = min(int(self.rank_samples), int(pos.numel()), int(neg.numel()))
                    pos_sel = pos[torch.randperm(pos.numel(), device=attn_logits.device)[:n]]
                    neg_sel = neg[torch.randperm(neg.numel(), device=attn_logits.device)[:n]]
                    diff = Aflat[b, pos_sel] - Aflat[b, neg_sel]
                    loss_rank = loss_rank + F.softplus(-diff).mean()

                loss_rank = loss_rank / float(max(1, B))

                loss_att = (loss_focal + self.rank_weight * loss_rank) * self.lambda_att / float(self.iter_nums)
'''

txt2 = re.sub(pattern, replacement, txt, flags=re.MULTILINE)
if txt2 == txt:
    raise RuntimeError('Could not patch the fp32 attention block (pattern not found).')
txt = txt2

p.write_text(txt, encoding='utf-8')
print('Patched ranking loss into:', p)

import prism3d.engine.interactive_runner_attn as ira
importlib.reload(ira)
from prism3d.engine.interactive_runner_attn import InteractiveRunner3D_Attn
print('Reloaded InteractiveRunner3D_Attn OK')

Patched ranking loss into: /content/prism3d_project/prism3d/engine/interactive_runner_attn.py
Reloaded InteractiveRunner3D_Attn OK


In [158]:
import torch
import torchio as tio
from torch.utils.data import DataLoader
from prism3d.data.dataset import PrismCTDataset
from prism3d.models.voxel_attention_head import VoxelAttentionHead3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/content/prism_data_tiny/colon'

val_tio = tio.Compose([tio.ToCanonical(), tio.Resample(1)])
ds = PrismCTDataset(dataset='colon', data_dir=DATA_ROOT, split='train', torchio_transform=val_tio)
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

img, lab, _ = next(iter(dl))

model = model.to(device).eval()

attn_head = VoxelAttentionHead3D(feat_ch=32, mid_ch=32, init_bias=-4.0, out_w_std=1e-3).to(device).train()
opt_attn = torch.optim.AdamW(attn_head.parameters(), lr=5e-4, weight_decay=1e-2)

runner_attn = InteractiveRunner3D_Attn(
    model=model,
    attention_head=attn_head,
    device=device,
    iter_nums=3,
    num_clicks_train=1,
    alpha_mix=0.7,
    band_w=3,
    lambda_att=5.0,
    topk=1024,
    min_dist=12,
    use_box=True,
)

loss_att = runner_attn.train_step(
    image=img,
    label=lab,
    opt=opt_attn,
    scaler=scaler,
    autocast_ctx=autocast_ctx,
    grad_clip=1.0,
    debug=True,
)

print('loss_att_sum:', loss_att)

/content/prism3d_project/prism3d/engine/interactive_runner_attn.py:137: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  


debug attn mean/max: 0.017988834530115128 0.018451889976859093
debug T mean/max   : 0.0010184765560552478 1.0
debug attn on T>0  : 0.01798563450574875
debug sampled zyx  : (59, 53, 76) attn@pt: 0.018213322386145592
loss_att_sum: 0.708106592297554
